<a href="https://www.kaggle.com/code/ab0y04/skin-lesion-imagenet?scriptVersionId=337462178" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import os
import pandas as pd

# HAM10000 metadata preview
ham_meta = pd.read_csv('/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')
print('\nHAM10000 metadata shape:', ham_meta.shape)
print('HAM10000 columns:', ham_meta.columns.tolist())
print(ham_meta.head(3))

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# -------------------------------------------------------
# PATHS
# -------------------------------------------------------
ISIC2019_IMG_DIR = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC2019_CSV = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'

HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'

# -------------------------------------------------------
# SHARED CLASS NAMES
# HAM10000 uses: bkl, nv, df, mel, vasc, bcc, akiec
# ISIC 2019 uses: BKL, NV, DF, MEL, VASC, BCC, AK, SCC
# We use the 7 classes both share, drop AK and SCC
# -------------------------------------------------------
CLASS_NAMES = ['MEL', 'NV', 'BCC', 'BKL', 'DF', 'VASC', 'mel', 'nv', 'bcc', 'bkl', 'df', 'vasc']

# Unified label map — both datasets mapped to same strings
LABEL_MAP = {
    # ISIC 2019
    'MEL': 'melanoma',
    'NV': 'nevus',
    'BCC': 'bcc',
    'BKL': 'bkl',
    'DF': 'df',
    'VASC': 'vasc',
    # HAM10000
    'mel': 'melanoma',
    'nv': 'nevus',
    'bcc': 'bcc',
    'bkl': 'bkl',
    'df': 'df',
    'vasc': 'vasc',
    'akiec': None  # drop this class, no equivalent in ISIC 2019
}

FINAL_CLASSES = ['melanoma', 'nevus', 'bcc', 'bkl', 'df', 'vasc']
NUM_CLASSES = len(FINAL_CLASSES)

print('Paths set.')
print(f'Number of shared classes: {NUM_CLASSES}')
print(f'Classes: {FINAL_CLASSES}')

In [ ]:
# -------------------------------------------------------
# BUILD ISIC 2019 DATAFRAME
# -------------------------------------------------------
isic_df = pd.read_csv(ISIC2019_CSV)

# Drop UNK, AK, SCC — not in HAM10000
isic_df = isic_df.drop(columns=['UNK', 'AK', 'SCC'], errors='ignore')

# Convert one-hot to single label
isic_cols = ['MEL', 'NV', 'BCC', 'BKL', 'DF', 'VASC']
isic_df['raw_label'] = isic_df[isic_cols].idxmax(axis=1)
isic_df['label'] = isic_df['raw_label'].map(LABEL_MAP)

# Build image path
isic_df['image_path'] = isic_df['image'].apply(
    lambda x: os.path.join(ISIC2019_IMG_DIR, x + '.jpg')
)

# Add source tag
isic_df['source'] = 'isic2019'

# Keep only existing files and valid labels
isic_df = isic_df[isic_df['image_path'].apply(os.path.exists)]
isic_df = isic_df[isic_df['label'].notna()]
isic_df = isic_df[['image_path', 'label', 'source']].reset_index(drop=True)

print(f'ISIC 2019: {len(isic_df)} images')
print(isic_df['label'].value_counts())

In [ ]:
# -------------------------------------------------------
# BUILD HAM10000 DATAFRAME
# -------------------------------------------------------
ham_df = pd.read_csv(HAM_CSV)

# Map HAM labels to unified labels
ham_df['label'] = ham_df['dx'].map(LABEL_MAP)

# Build image path — images split across two folders
def find_ham_image(image_id):
    for folder in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        path = os.path.join(folder, image_id + '.jpg')
        if os.path.exists(path):
            return path
    return None

ham_df['image_path'] = ham_df['image_id'].apply(find_ham_image)

# Add source tag
ham_df['source'] = 'ham10000'

# Keep only existing files and valid labels
ham_df = ham_df[ham_df['image_path'].notna()]
ham_df = ham_df[ham_df['label'].notna()]
ham_df = ham_df[['image_path', 'label', 'source']].reset_index(drop=True)

print(f'HAM10000: {len(ham_df)} images')
print(ham_df['label'].value_counts())

In [ ]:
# -------------------------------------------------------
# COMBINE INTO MASTER DATAFRAME
# -------------------------------------------------------
master_df = pd.concat([isic_df, ham_df], ignore_index=True)

print(f'Total images: {len(master_df)}')
print(f'\nBy source:')
print(master_df['source'].value_counts())
print(f'\nBy label:')
print(master_df['label'].value_counts())
print(f'\nBy source and label:')
print(master_df.groupby(['source', 'label']).size().unstack(fill_value=0))

In [ ]:
# -------------------------------------------------------
# SPLIT STRATEGY
# Split each source separately, then combine
# This ensures both sources are represented in train/val/test
# -------------------------------------------------------

from sklearn.model_selection import train_test_split

def split_source(df, test_size=0.15, val_size=0.15, random_state=42):
    train, temp = train_test_split(
        df, test_size=test_size + val_size,
        stratify=df['label'], random_state=random_state
    )
    val, test = train_test_split(
        temp, test_size=0.5,
        stratify=temp['label'], random_state=random_state
    )
    return train, val, test

# Split each source independently
isic_train, isic_val, isic_test = split_source(isic_df)
ham_train, ham_val, ham_test = split_source(ham_df)

# Combine splits
train_df = pd.concat([isic_train, ham_train], ignore_index=True)
val_df = pd.concat([isic_val, ham_val], ignore_index=True)
test_df = pd.concat([isic_test, ham_test], ignore_index=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'\nTrain source breakdown:')
print(train_df['source'].value_counts())
print(f'\nTest label distribution:')
print(test_df['label'].value_counts())

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights
# This tells the model to penalize mistakes on rare classes more heavily
# Without this, the model will just predict nevus for everything and get 56% accuracy

classes = np.array(FINAL_CLASSES)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['label']
)

class_weight_dict = dict(zip(classes, class_weights_array))

print('Class weights:')
for cls, weight in sorted(class_weight_dict.items(), key=lambda x: x[1]):
    print(f'  {cls}: {weight:.3f}')

print('\nHigher weight = model penalized more for missing that class')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30

def build_generators(preprocess_fn=None):
    if preprocess_fn is not None:
        train_gen = ImageDataGenerator(
            preprocessing_function=preprocess_fn,
            rotation_range=20,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1
        )
        val_test_gen = ImageDataGenerator(preprocessing_function=preprocess_fn)
    else:
        train_gen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1
        )
        val_test_gen = ImageDataGenerator(rescale=1./255)

    train_flow = train_gen.flow_from_dataframe(
        train_df, x_col='image_path', y_col='label',
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=True, seed=42
    )
    val_flow = val_test_gen.flow_from_dataframe(
        val_df, x_col='image_path', y_col='label',
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False
    )
    test_flow = val_test_gen.flow_from_dataframe(
        test_df, x_col='image_path', y_col='label',
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False
    )
    return train_flow, val_flow, test_flow

print('Generator function ready.')
print(f'IMG_SIZE: {IMG_SIZE} | BATCH_SIZE: {BATCH_SIZE} | EPOCHS: {EPOCHS}')

In [ ]:
# -------------------------------------------------------
# MODEL 1: Custom CNN (from scratch, no pretrained weights)
# -------------------------------------------------------
def build_custom_cnn(num_classes, input_shape=(224, 224, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Classification head
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='custom_cnn')
    return model


# -------------------------------------------------------
# MODEL 2: EfficientNetB0 (ImageNet pretrained)
# -------------------------------------------------------
def build_efficientnet(num_classes, input_shape=(224, 224, 3)):
    backbone = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    backbone.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name='efficientnet_b0')
    return model, backbone


# Build both
custom_cnn = build_custom_cnn(NUM_CLASSES)
efficientnet_model, backbone = build_efficientnet(NUM_CLASSES)

print(f'Custom CNN parameters: {custom_cnn.count_params():,}')
print(f'EfficientNetB0 parameters: {efficientnet_model.count_params():,}')

In [ ]:
# -------------------------------------------------------
# TRAIN EFFICIENTNETB0 — Phase 1 (head only)
# -------------------------------------------------------
#eff_train, eff_val, eff_test = build_generators(preprocess_fn=efficientnet_preprocess)

#efficientnet_model.compile(
    #optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    #loss='categorical_crossentropy',
    #metrics=['accuracy']
#)

#print('Phase 1: Training head only (backbone frozen)...')
#history_eff_phase1 = efficientnet_model.fit(
    #eff_train,
    #validation_data=eff_val,
    #epochs=10,
    #class_weight=indexed_class_weights,
    #verbose=1
#)
#print('Phase 1 done.')

In [ ]:
# -------------------------------------------------------
# TRAIN EFFICIENTNETB0 — Phase 2 (full fine-tuning)
# -------------------------------------------------------
#backbone.trainable = True

#efficientnet_model.compile(
    #optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    #loss='categorical_crossentropy',
    #metrics=['accuracy']
#)

#callbacks_eff = [
    #EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    #ModelCheckpoint('best_efficientnet.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    #CSVLogger('efficientnet_training_log.csv')
#]

#print('Phase 2: Fine-tuning full model (backbone unfrozen)...')
#history_eff_phase2 = efficientnet_model.fit(
   # eff_train,
    #validation_data=eff_val,
    #epochs=30,
    #class_weight=indexed_class_weights,
    #callbacks=callbacks_eff,
    #verbose=1
#)
#print('Phase 2 done.')

In [ ]:
'''cnn_train, cnn_val, cnn_test = build_generators(preprocess_fn=None)
eff_train, eff_val, eff_test = build_generators(preprocess_fn=efficientnet_preprocess)
class_indices = cnn_train.class_indices
indexed_class_weights = {class_indices[k]: v for k, v in class_weight_dict.items()}
print('Generators ready.')'''

In [ ]:
''' import tensorflow as tf
from tensorflow import keras

custom_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Models compiled.')'''

In [ ]:
#cnn_test.reset()
#eff_test.reset()

#cnn_loss_val, cnn_acc_val = custom_cnn.evaluate(cnn_test, verbose=0)
#eff_loss_val, eff_acc_val = efficientnet_model.evaluate(eff_test, verbose=0)

#print('='*50)
#print('FINAL TEST SET RESULTS')
#print('='*50)
#print(f'Custom CNN      : {cnn_acc_val*100:.2f}%')
#print(f'EfficientNetB0  : {eff_acc_val*100:.2f}%')
#print(f'Gap             : {(eff_acc_val - cnn_acc_val)*100:.2f} percentage points')
#print('='*50)

In [ ]:
#from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

#backbone.trainable = True

#efficientnet_model.compile(
    #optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    #loss='categorical_crossentropy',
    #metrics=['accuracy']
#)

#callbacks_eff = [
    #EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    #ModelCheckpoint('/kaggle/working/best_efficientnet_v2.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    #CSVLogger('/kaggle/working/efficientnet_v2_log.csv')
#]

#print('Continuing EfficientNetB0 training...')
#history_eff_continued = efficientnet_model.fit(
    #eff_train,
    #validation_data=eff_val,
    #epochs=20,
    #class_weight=indexed_class_weights,
    #callbacks=callbacks_eff,
    #verbose=1
#)
#print('Done. Best val accuracy:', max(history_eff_continued.history['val_accuracy']))

In [ ]:
#callbacks_eff = [
    #EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    #ModelCheckpoint('/kaggle/working/best_efficientnet_v3.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
   # CSVLogger('/kaggle/working/efficientnet_v3_log.csv')
#]

#print('Continuing EfficientNetB0 training...')
#history_eff_v3 = efficientnet_model.fit(
    #eff_train,
    #validation_data=eff_val,
    #epochs=20,
    #class_weight=indexed_class_weights,
    #callbacks=callbacks_eff,
    #verbose=1
#)
#print('Done. Best val accuracy:', max(history_eff_v3.history['val_accuracy']))

In [ ]:
#eff_test.reset()
#cnn_test.reset()

#cnn_loss_val, cnn_acc_val = custom_cnn.evaluate(cnn_test, verbose=0)
#eff_loss_val, eff_acc_val = efficientnet_model.evaluate(eff_test, verbose=0)

#print('='*50)
#print('FINAL TEST SET RESULTS')
#print('='*50)
#print(f'Custom CNN      : {cnn_acc_val*100:.2f}%')
#print(f'EfficientNetB0  : {eff_acc_val*100:.2f}%')
#print(f'Gap             : {(eff_acc_val - cnn_acc_val)*100:.2f} percentage points')
#print('='*50)

In [ ]:
'''custom_cnn.load_weights('/kaggle/input/datasets/ab0y04/best-custom-cnn/best_custom_cnn.keras')
efficientnet_model.load_weights('/kaggle/input/notebooks/ab0y04/skin-lession-cnn-efficientnetb0/best_efficientnet_v3.keras')
print('Both models loaded.')'''

In [ ]:
'''cnn_train, cnn_val, cnn_test = build_generators(preprocess_fn=None)
eff_train, eff_val, eff_test = build_generators(preprocess_fn=efficientnet_preprocess)
class_indices = cnn_train.class_indices
indexed_class_weights = {class_indices[k]: v for k, v in class_weight_dict.items()}

custom_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Generators and compile done.')'''

In [ ]:
#from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

#backbone.trainable = True

#efficientnet_model.compile(
    #optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    #loss='categorical_crossentropy',
    #metrics=['accuracy']
#)

#callbacks_eff = [
    #EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    #ModelCheckpoint('/kaggle/working/best_efficientnet_v4.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    #CSVLogger('/kaggle/working/efficientnet_v4_log.csv')
#]

#print('Continuing EfficientNetB0 training...')
#history_eff_v4 = efficientnet_model.fit(
    #eff_train,
    #validation_data=eff_val,
    #epochs=20,
    #class_weight=indexed_class_weights,
    #callbacks=callbacks_eff,
    #verbose=1
#)
#print('Done. Best val accuracy:', max(history_eff_v4.history['val_accuracy']))

In [ ]:
''' eff_test.reset()
eff_loss_val, eff_acc_val = efficientnet_model.evaluate(eff_test, verbose=0)
print(f'EfficientNetB0 Final Test Accuracy: {eff_acc_val*100:.2f}%')
print(f'Custom CNN Test Accuracy: 52.84%')
print(f'Gap: {(eff_acc_val - 0.5284)*100:.2f} percentage points')'''

In [ ]:
''' from sklearn.metrics import classification_report
import numpy as np

cnn_test.reset()
eff_test.reset()

cnn_preds = custom_cnn.predict(cnn_test, verbose=0)
eff_preds = efficientnet_model.predict(eff_test, verbose=0)

cnn_pred_classes = np.argmax(cnn_preds, axis=1)
eff_pred_classes = np.argmax(eff_preds, axis=1)
true_classes = cnn_test.classes

class_indices = cnn_test.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print('CUSTOM CNN:')
print(classification_report(true_classes, cnn_pred_classes, target_names=target_names))
print('EFFICIENTNETB0:')
print(classification_report(true_classes, eff_pred_classes, target_names=target_names)) '''

In [ ]:
''' import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, roc_auc_score, 
                              roc_curve, auc)
from sklearn.preprocessing import label_binarize
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------
# GET PREDICTIONS (already have these from classification report)
# Reset and re-predict to be safe
# -------------------------------------------------------
cnn_test.reset()
eff_test.reset()

cnn_preds_prob = custom_cnn.predict(cnn_test, verbose=0)
eff_preds_prob = efficientnet_model.predict(eff_test, verbose=0)

cnn_pred_classes = np.argmax(cnn_preds_prob, axis=1)
eff_pred_classes = np.argmax(eff_preds_prob, axis=1)
true_classes = cnn_test.classes

class_indices = cnn_test.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

# Binarize true labels for AUC computation
true_binarized = label_binarize(true_classes, classes=list(range(NUM_CLASSES)))

print('Predictions ready.')
print(f'Test set size: {len(true_classes)}')
print(f'Classes: {target_names}') '''

In [ ]:
# -------------------------------------------------------
# CONFUSION MATRICES
# -------------------------------------------------------
#fig, axes = plt.subplots(1, 2, figsize=(16, 7))

#for ax, preds, title in zip(
    #axes,
    #[cnn_pred_classes, eff_pred_classes],
    #['Custom CNN (52.84%)', 'EfficientNetB0 (82.98%)']
#):
    #cm = confusion_matrix(true_classes, preds)
    
    # Normalize by row (true label) to show recall per class
    #cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    #sns.heatmap(
        #cm_norm, annot=True, fmt='.2f', ax=ax,
        #xticklabels=target_names, yticklabels=target_names,
        #cmap='Blues', vmin=0, vmax=1
    #)
    #ax.set_title(f'{title}', fontsize=13, fontweight='bold')
    #ax.set_ylabel('True Label')
    #ax.set_xlabel('Predicted Label')
    #plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

#plt.suptitle('Normalized Confusion Matrices — Skin Lesion Classification\n(Values show recall per class)',
             #fontsize=13, fontweight='bold')
#plt.tight_layout()
#plt.savefig('/kaggle/working/confusion_matrices.png', dpi=150, bbox_inches='tight')
#plt.show()
#print('Saved: confusion_matrices.png')

In [ ]:
#true_classes_np = np.array(true_classes)

#print('AUC with DeLong 95% Confidence Intervals')
#print('='*65)
#print(f'{"Class":<12} {"CNN AUC":>10} {"CNN 95% CI":>18} {"Eff AUC":>10} {"Eff 95% CI":>18}')
#print('-'*65)

#cnn_aucs, eff_aucs = [], []

#for i, cls in enumerate(target_names):
    #y_true_bin = (true_classes_np == i).astype(int)
    
    #cnn_auc, cnn_lo, cnn_hi = delong_ci(y_true_bin, cnn_preds_prob[:, i])
    #eff_auc, eff_lo, eff_hi = delong_ci(y_true_bin, eff_preds_prob[:, i])
    
    #cnn_aucs.append(cnn_auc)
    #eff_aucs.append(eff_auc)
    
    #print(f'{cls:<12} {cnn_auc:>10.3f} {f"({cnn_lo:.3f}-{cnn_hi:.3f})":>18} '
          #f'{eff_auc:>10.3f} {f"({eff_lo:.3f}-{eff_hi:.3f})":>18}')

#print('-'*65)
#print(f'{"Macro avg":<12} {np.mean(cnn_aucs):>10.3f} {"":>18} {np.mean(eff_aucs):>10.3f}')
#print('='*65)

In [ ]:
# -------------------------------------------------------
# MCNEMAR'S TEST
# Tests whether the difference in errors between models
# is statistically significant
# -------------------------------------------------------
#from statsmodels.stats.contingency_tables import mcnemar

#cnn_correct = (cnn_pred_classes == true_classes_np)
#eff_correct = (eff_pred_classes == true_classes_np)

# Contingency table
# n00: both wrong, n01: CNN wrong EfficientNet right
# n10: CNN right EfficientNet wrong, n11: both right
#n00 = np.sum(~cnn_correct & ~eff_correct)
#n01 = np.sum(~cnn_correct & eff_correct)
#n10 = np.sum(cnn_correct & ~eff_correct)
#n11 = np.sum(cnn_correct & eff_correct)

#table = [[n11, n10], [n01, n00]]
#result = mcnemar(table, exact=False, correction=True)

#print('McNemar\'s Test: Custom CNN vs EfficientNetB0')
#print('='*50)
#print(f'Both correct (n11):           {n11}')
#print(f'CNN correct, Eff wrong (n10): {n10}')
#print(f'CNN wrong, Eff correct (n01): {n01}')
#print(f'Both wrong (n00):             {n00}')
#print('-'*50)
#print(f'McNemar statistic: {result.statistic:.4f}')
#print(f'p-value:           {result.pvalue:.2e}')
#print(f'Significant (p<0.05): {result.pvalue < 0.05}')
#print('='*50)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

def build_mobilenetv2(num_classes, input_shape=(224, 224, 3)):
    backbone = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    backbone.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name='mobilenetv2')
    return model, backbone

mobilenet_model, mobilenet_backbone = build_mobilenetv2(NUM_CLASSES)
print(f'MobileNetV2 ready. Parameters: {mobilenet_model.count_params():,}')

In [ ]:
'''
# MobileNetV2 Phase 1 - head only
mob_train, mob_val, mob_test = build_generators(preprocess_fn=mobilenet_preprocess)

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 1: MobileNetV2 head only...')
history_mob_phase1 = mobilenet_model.fit(
    mob_train,
    validation_data=mob_val,
    epochs=10,
    class_weight=indexed_class_weights,
    verbose=1
)
print('Phase 1 done. Best val accuracy:', max(history_mob_phase1.history['val_accuracy'])) '''

In [ ]:
""" # MobileNetV2 Phase 2 - full fine-tuning
mobilenet_backbone.trainable = True

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_mob = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/best_mobilenet.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    CSVLogger('/kaggle/working/mobilenet_log.csv')
]

print('Phase 2: MobileNetV2 full fine-tuning...')
history_mob_phase2 = mobilenet_model.fit(
    mob_train,
    validation_data=mob_val,
    epochs=30,
    class_weight=indexed_class_weights,
    callbacks=callbacks_mob,
    verbose=1
)
print('Phase 2 done. Best val accuracy:', max(history_mob_phase2.history['val_accuracy'])) """

In [ ]:
""" callbacks_mob = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/best_mobilenet_v2.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    CSVLogger('/kaggle/working/mobilenet_v2_log.csv')
]

print('Continuing MobileNetV2...')
history_mob_v2 = mobilenet_model.fit(
    mob_train,
    validation_data=mob_val,
    epochs=30,
    class_weight=indexed_class_weights,
    callbacks=callbacks_mob,
    verbose=1
)
print('Done. Best val accuracy:', max(history_mob_v2.history['val_accuracy'])) """

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input/notebooks'):
    for f in files:
        if 'mobilenet' in f.lower():
            print(os.path.join(root, f))

In [ ]:
'''
mobilenet_model.load_weights('/kaggle/input/notebooks/ab0y04/skin-lession-cnn-efficientnetb0/best_mobilenet.keras')
print('MobileNetV2 loaded.')
'''

In [ ]:
'''
mob_train, mob_val, mob_test = build_generators(preprocess_fn=mobilenet_preprocess)
class_indices = mob_train.class_indices
indexed_class_weights = {class_indices[k]: v for k, v in class_weight_dict.items()}

mobilenet_backbone.trainable = True

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_mob = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/best_mobilenet_v2.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    CSVLogger('/kaggle/working/mobilenet_v2_log.csv')
]

print('Continuing MobileNetV2...')
history_mob_cont = mobilenet_model.fit(
    mob_train,
    validation_data=mob_val,
    epochs=30,
    class_weight=indexed_class_weights,
    callbacks=callbacks_mob,
    verbose=1
)
print('Done. Best val accuracy:', max(history_mob_cont.history['val_accuracy'])) 
'''

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input/notebooks'):
    for f in files:
        if 'mobilenet' in f.lower():
            print(os.path.join(root, f))

In [ ]:
'''
mob_test.reset()
mob_loss, mob_acc = mobilenet_model.evaluate(mob_test, verbose=0)
print(f'MobileNetV2 Test Accuracy: {mob_acc*100:.2f}%')
print(f'Custom CNN: 52.84%')
print(f'Gap: {(mob_acc - 0.5284)*100:.2f} percentage points') 
'''

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def build_resnet50(num_classes, input_shape=(224, 224, 3)):
    backbone = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    backbone.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name='resnet50')
    return model, backbone

resnet_model, resnet_backbone = build_resnet50(NUM_CLASSES)
print(f'ResNet50 ready. Parameters: {resnet_model.count_params():,}')

In [ ]:
'''
res_train, res_val, res_test = build_generators(preprocess_fn=resnet_preprocess)

resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 1: ResNet50 head only...')
history_res_phase1 = resnet_model.fit(
    res_train,
    validation_data=res_val,
    epochs=10,
    class_weight=indexed_class_weights,
    verbose=1
)
print('Phase 1 done. Best val accuracy:', max(history_res_phase1.history['val_accuracy']))
'''

In [ ]:
'''
resnet_backbone.trainable = True

resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_res = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/best_resnet50.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    CSVLogger('/kaggle/working/resnet50_log.csv')
]

print('Phase 2: ResNet50 full fine-tuning...')
history_res_phase2 = resnet_model.fit(
    res_train,
    validation_data=res_val,
    epochs=30,
    class_weight=indexed_class_weights,
    callbacks=callbacks_res,
    verbose=1
)
print('Phase 2 done. Best val accuracy:', max(history_res_phase2.history['val_accuracy']))
'''

In [ ]:
resnet_model.load_weights('/kaggle/input/notebooks/ab0y04/skin-lession-cnn-efficientnetb0/best_resnet50.keras')
print('ResNet50 loaded.')

res_train, res_val, res_test = build_generators(preprocess_fn=resnet_preprocess)
class_indices = res_train.class_indices
indexed_class_weights = {class_indices[k]: v for k, v in class_weight_dict.items()}

resnet_backbone.trainable = True
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Setup complete.')

In [ ]:
callbacks_res = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/best_resnet50_v2.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    CSVLogger('/kaggle/working/resnet50_v2_log.csv')
]

print('Continuing ResNet50...')
history_res_cont = resnet_model.fit(
    res_train,
    validation_data=res_val,
    epochs=20,
    class_weight=indexed_class_weights,
    callbacks=callbacks_res,
    verbose=1
)
print('Done. Best val accuracy:', max(history_res_cont.history['val_accuracy']))

In [ ]:
res_test.reset()
res_loss, res_acc = resnet_model.evaluate(res_test, verbose=0)
print(f'ResNet50 Test Accuracy: {res_acc*100:.2f}%')
print(f'Custom CNN: 52.84%')
print(f'Gap: {(res_acc - 0.5284)*100:.2f} percentage points')

In [ ]:
mob_train, mob_val, mob_test = build_generators(preprocess_fn=mobilenet_preprocess)
print('Done.')

In [ ]:
from sklearn.preprocessing import label_binarize

class_indices = mob_test.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print('MOBILENETV2:')
print(classification_report(true_classes_np, mob_pred_classes, target_names=target_names))

print('RESNET50:')
print(classification_report(true_classes_np, res_pred_classes, target_names=target_names))

mob_auc = roc_auc_score(label_binarize(true_classes_np, classes=list(range(NUM_CLASSES))), mob_preds_prob, average='macro')
res_auc = roc_auc_score(label_binarize(true_classes_np, classes=list(range(NUM_CLASSES))), res_preds_prob, average='macro')

print(f'MobileNetV2 Macro AUC: {mob_auc:.3f}')
print(f'ResNet50 Macro AUC: {res_auc:.3f}')

In [ ]:
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mob_test.reset()
mob_loss, mob_acc = mobilenet_model.evaluate(mob_test, verbose=0)
print(f'MobileNetV2 re-check accuracy: {mob_acc*100:.2f}%')

In [ ]:
mobilenet_model.load_weights('/kaggle/input/notebooks/ab0y04/skin-lession-cnn-efficientnetb0/best_mobilenet_v2.keras')
mob_test.reset()
mob_loss, mob_acc = mobilenet_model.evaluate(mob_test, verbose=0)
print(f'MobileNetV2 re-check accuracy: {mob_acc*100:.2f}%')

In [ ]:
res_test.reset()
res_loss, res_acc = resnet_model.evaluate(res_test, verbose=0)
print(f'ResNet50 re-check accuracy: {res_acc*100:.2f}%')

In [ ]:
mob_test.reset()
res_test.reset()

mob_preds_prob = mobilenet_model.predict(mob_test, verbose=0)
res_preds_prob = resnet_model.predict(res_test, verbose=0)

mob_pred_classes = np.argmax(mob_preds_prob, axis=1)
res_pred_classes = np.argmax(res_preds_prob, axis=1)
true_classes_np = np.array(mob_test.classes)

print('MOBILENETV2:')
print(classification_report(true_classes_np, mob_pred_classes, target_names=target_names))

print('RESNET50:')
print(classification_report(true_classes_np, res_pred_classes, target_names=target_names))

mob_auc = roc_auc_score(label_binarize(true_classes_np, classes=list(range(NUM_CLASSES))), mob_preds_prob, average='macro')
res_auc = roc_auc_score(label_binarize(true_classes_np, classes=list(range(NUM_CLASSES))), res_preds_prob, average='macro')

print(f'MobileNetV2 Macro AUC: {mob_auc:.3f}')
print(f'ResNet50 Macro AUC: {res_auc:.3f}')

In [ ]:
import os

# List all files in the dataset
for root, dirs, files in os.walk('/kaggle/input/datasets/ab0y04/all-best-models'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
custom_cnn.load_weights('/kaggle/input/datasets/ab0y04/all-best-models/best_custom_cnn.keras')
efficientnet_model.load_weights('/kaggle/input/datasets/ab0y04/all-best-models/best_efficientnet_v4.keras')
mobilenet_model.load_weights('/kaggle/input/datasets/ab0y04/all-best-models/best_mobilenet_v2.keras')
resnet_model.load_weights('/kaggle/input/datasets/ab0y04/all-best-models/best_resnet50_v2.keras')

custom_cnn.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
efficientnet_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
mobilenet_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
resnet_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

cnn_train, cnn_val, cnn_test = build_generators(preprocess_fn=None)
eff_train, eff_val, eff_test = build_generators(preprocess_fn=efficientnet_preprocess)
mob_train, mob_val, mob_test = build_generators(preprocess_fn=mobilenet_preprocess)
res_train, res_val, res_test = build_generators(preprocess_fn=resnet_preprocess)

for name, model, test_gen in [('Custom CNN', custom_cnn, cnn_test), 
                                ('EfficientNetB0', efficientnet_model, eff_test),
                                ('MobileNetV2', mobilenet_model, mob_test),
                                ('ResNet50', resnet_model, res_test)]:
    test_gen.reset()
    loss, acc = model.evaluate(test_gen, verbose=0)
    print(f'{name}: {acc*100:.2f}%')

In [ ]:
cnn_last_conv = [l.name for l in custom_cnn.layers if isinstance(l, layers.Conv2D)][-1]
mob_last_conv = 'Conv_1'
eff_last_conv = 'top_conv'
res_last_conv = 'conv5_block3_out'

print('CNN last conv:', cnn_last_conv)
print('MobileNetV2 last conv:', mob_last_conv)
print('EfficientNetB0 last conv:', eff_last_conv)
print('ResNet50 last conv:', res_last_conv)

# Verify layers exist
for name, model, layer_name in [('CNN', custom_cnn, cnn_last_conv),
                                  ('MobileNetV2', mobilenet_model, mob_last_conv),
                                  ('EfficientNetB0', efficientnet_model, eff_last_conv),
                                  ('ResNet50', resnet_model, res_last_conv)]:
    try:
        model.get_layer(layer_name)
        print(f'{name}: OK')
    except:
        print(f'{name}: LAYER NOT FOUND - {layer_name}')

In [ ]:
print('MobileNetV2 backbone layers (last 5):')
for l in mobilenet_backbone.layers[-5:]:
    print(' ', l.name)

print('\nEfficientNetB0 backbone layers (last 5):')
for l in backbone.layers[-5:]:
    print(' ', l.name)

print('\nResNet50 backbone layers (last 5):')
for l in resnet_backbone.layers[-5:]:
    print(' ', l.name)

In [ ]:
try:
    test_model = models.Model(
        inputs=efficientnet_model.inputs,
        outputs=[backbone.get_layer('top_conv').output, efficientnet_model.output]
    )
    print('Nested model construction: OK')
except Exception as e:
    print('Error:', e)

In [ ]:
import cv2

In [ ]:
# Build custom_cnn's graph by calling it once
dummy_input = np.zeros((1, 224, 224, 3), dtype=np.float32)
_ = custom_cnn(dummy_input)
print('custom_cnn built.')

In [ ]:
# Rebuild custom_cnn as functional model
inputs_cnn = layers.Input(shape=(224, 224, 3))
x = inputs_cnn
for layer in custom_cnn.layers:
    x = layer(x)
custom_cnn_functional = models.Model(inputs_cnn, x, name='custom_cnn_functional')
custom_cnn_functional.set_weights(custom_cnn.get_weights())

print('Functional version built.')
print(custom_cnn_functional.inputs)

In [ ]:
def get_gradcam_heatmap_nested_v2(full_model, backbone_model, head_layers, img_array, last_conv_layer_name):
    """
    head_layers: list of layers AFTER the backbone in full_model 
                  (GlobalAveragePooling2D, Dense, Dropout, Dense)
    """
    # Sub-model: backbone input -> last conv layer output
    conv_layer_model = models.Model(
        inputs=backbone_model.inputs,
        outputs=backbone_model.get_layer(last_conv_layer_name).output
    )
    # Sub-model: backbone input -> backbone final output (for continuing through head)
    backbone_output_model = models.Model(
        inputs=backbone_model.inputs,
        outputs=backbone_model.output
    )

    img_tensor = tf.convert_to_tensor(img_array)
    with tf.GradientTape() as tape:
        conv_output = conv_layer_model(img_tensor)
        tape.watch(conv_output)
        
        backbone_out = backbone_output_model(img_tensor)
        x = backbone_out
        for layer in head_layers:
            x = layer(x)
        predictions = x
        pred_class = tf.argmax(predictions[0])
        class_score = predictions[:, pred_class]

    grads = tape.gradient(class_score, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_class.numpy()

# Head layers = everything in full_model after the backbone
# Index 0 is Input, index 1 is backbone, rest is head
eff_head_layers = efficientnet_model.layers[2:]
mob_head_layers = mobilenet_model.layers[2:]
res_head_layers = resnet_model.layers[2:]

print('Eff head layers:', [l.name for l in eff_head_layers])

In [ ]:
def get_gradcam_heatmap_nested_v3(full_model, backbone_model, head_layers, img_array, last_conv_layer_name):
    dual_output_model = models.Model(
        inputs=backbone_model.inputs,
        outputs=[backbone_model.get_layer(last_conv_layer_name).output, backbone_model.output]
    )
    img_tensor = tf.convert_to_tensor(img_array)
    with tf.GradientTape() as tape:
        conv_output, backbone_out = dual_output_model(img_tensor)
        tape.watch(conv_output)
        x = backbone_out
        for layer in head_layers:
            x = layer(x)
        predictions = x
        pred_class = tf.argmax(predictions[0])
        class_score = predictions[:, pred_class]
    grads = tape.gradient(class_score, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_class.numpy()


def show_gradcam_row(image_path, true_label):
    img_orig = cv2.imread(image_path)
    img_orig = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)
    img_orig = cv2.resize(img_orig, (IMG_SIZE, IMG_SIZE))

    img_cnn = np.expand_dims(img_orig.astype(np.float32) / 255.0, axis=0)
    img_eff = efficientnet_preprocess(np.expand_dims(img_orig.astype(np.float32), axis=0))
    img_mob = mobilenet_preprocess(np.expand_dims(img_orig.astype(np.float32), axis=0))
    img_res = resnet_preprocess(np.expand_dims(img_orig.astype(np.float32), axis=0))

    cnn_hm, cnn_pred = get_gradcam_heatmap_sequential(custom_cnn, img_cnn, cnn_conv_index)
    eff_hm, eff_pred = get_gradcam_heatmap_nested_v3(efficientnet_model, backbone, eff_head_layers, img_eff, 'top_conv')
    mob_hm, mob_pred = get_gradcam_heatmap_nested_v3(mobilenet_model, mobilenet_backbone, mob_head_layers, img_mob, 'Conv_1')
    res_hm, res_pred = get_gradcam_heatmap_nested_v3(resnet_model, resnet_backbone, res_head_layers, img_res, 'conv5_block3_out')

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(img_orig)
    axes[0].set_title(f'Original\nTrue: {true_label}', fontsize=10)
    axes[0].axis('off')

    for ax, hm, pred, name in zip(axes[1:],
                                    [cnn_hm, mob_hm, eff_hm, res_hm],
                                    [cnn_pred, mob_pred, eff_pred, res_pred],
                                    ['Custom CNN', 'MobileNetV2', 'EfficientNetB0', 'ResNet50']):
        overlay = overlay_heatmap(hm, img_orig)
        ax.imshow(overlay)
        ax.set_title(f'{name}\nPred: {idx_to_class.get(pred, pred)}', fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    return fig

for cls in ['melanoma', 'bcc', 'df']:
    sample = test_df[test_df['label'] == cls].iloc[0]
    fig = show_gradcam_row(sample['image_path'], cls)
    fig.suptitle(f'Grad-CAM Comparison: {cls.upper()}', fontsize=13, fontweight='bold', y=1.05)
    plt.savefig(f'/kaggle/working/gradcam_{cls}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: gradcam_{cls}.png')

In [ ]:
cnn_test.reset()
mob_test.reset()
eff_test.reset()
res_test.reset()

cnn_preds_prob = custom_cnn.predict(cnn_test, verbose=0)
mob_preds_prob = mobilenet_model.predict(mob_test, verbose=0)
eff_preds_prob = efficientnet_model.predict(eff_test, verbose=0)
res_preds_prob = resnet_model.predict(res_test, verbose=0)

cnn_pred_classes = np.argmax(cnn_preds_prob, axis=1)
mob_pred_classes = np.argmax(mob_preds_prob, axis=1)
eff_pred_classes = np.argmax(eff_preds_prob, axis=1)
res_pred_classes = np.argmax(res_preds_prob, axis=1)

true_classes_np = np.array(cnn_test.classes)

class_indices = cnn_test.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print('Predictions regenerated.')

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

models_data = [
    (cnn_pred_classes, 'Custom CNN'),
    (mob_pred_classes, 'MobileNetV2'),
    (eff_pred_classes, 'EfficientNetB0'),
    (res_pred_classes, 'ResNet50'),
]

for ax, (preds, title) in zip(axes, models_data):
    acc = (preds == true_classes_np).mean() * 100
    cm_ = confusion_matrix(true_classes_np, preds)
    cm_norm = cm_.astype('float') / cm_.sum(axis=1)[:, np.newaxis]

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', ax=ax,
        xticklabels=target_names, yticklabels=target_names,
        cmap='Blues', vmin=0, vmax=1, cbar=False
    )
    ax.set_title(f'{title} ({acc:.2f}%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    if ax == axes[0]:
        ax.set_ylabel('True')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.suptitle('Normalized Confusion Matrices — All 4 Models (Skin Lesion Classification)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices_all4.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices_all4.png')

In [ ]:
import os, random
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Seeds set: {SEED}")
print(f"TF version: {tf.__version__}")

In [ ]:
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras import Model, Input

NUM_CLASSES = 6

def build_pretrained_model(base_fn, input_shape=(224, 224, 3)):
    inputs = Input(shape=input_shape)
    base = base_fn(include_top=False, weights=None, input_tensor=inputs)
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs=inputs, outputs=outputs), base

def build_custom_cnn(input_shape=(224, 224, 3)):
    inputs = Input(shape=input_shape)
    x = inputs
    for filters in [32, 64, 128]:
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(0.25)(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs=inputs, outputs=outputs)

# Build
custom_cnn = build_custom_cnn()
efficientnet_model, eff_base = build_pretrained_model(EfficientNetB0)
mobilenet_model, mob_base = build_pretrained_model(MobileNetV2)
resnet_model, res_base = build_pretrained_model(ResNet50)

# Load weights
weight_path = '/kaggle/input/datasets/ab0y04/all-best-models/'
custom_cnn.load_weights(weight_path + 'best_custom_cnn.keras')
efficientnet_model.load_weights(weight_path + 'best_efficientnet_v4.keras')
mobilenet_model.load_weights(weight_path + 'best_mobilenet_v2.keras')
resnet_model.load_weights(weight_path + 'best_resnet50_v2.keras')

# Compile all
for m in [custom_cnn, efficientnet_model, mobilenet_model, resnet_model]:
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("All models built, weights loaded, compiled.")

In [ ]:
from tensorflow.keras.models import load_model

weight_path = '/kaggle/input/datasets/ab0y04/all-best-models/'

custom_cnn = load_model(weight_path + 'best_custom_cnn.keras')
efficientnet_model = load_model(weight_path + 'best_efficientnet_v4.keras')
mobilenet_model = load_model(weight_path + 'best_mobilenet_v2.keras')
resnet_model = load_model(weight_path + 'best_resnet50_v2.keras')

for name, m in [('Custom CNN', custom_cnn), ('EfficientNetB0', efficientnet_model),
                ('MobileNetV2', mobilenet_model), ('ResNet50', resnet_model)]:
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    print(f"{name}: {m.count_params():,} params")

In [ ]:
import pandas as pd
ISIC2019_CSV = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
raw = pd.read_csv(ISIC2019_CSV)

print("True ISIC counts:")
print(raw[['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']].sum().astype(int))

drop = raw.drop(columns=['UNK','AK','SCC'], errors='ignore')
cols = ['MEL','NV','BCC','BKL','DF','VASC']
print("\nWhat the prep code actually produced (MEL inflated):")
print(drop[cols].idxmax(axis=1).value_counts())

In [ ]:
import os, pandas as pd

meta_path = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
print("Exists:", os.path.exists(meta_path))

if os.path.exists(meta_path):
    meta = pd.read_csv(meta_path)
    print("Columns:", list(meta.columns))
    print("Rows:", len(meta))
    print(meta.head(5).to_string())

In [ ]:
import pandas as pd

meta_path = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
meta = pd.read_csv(meta_path)

has_lesion = meta['lesion_id'].notna()
print("Total ISIC images:", len(meta))
print("Images WITH a lesion_id (likely HAM origin):", has_lesion.sum())
print("Images WITHOUT a lesion_id (likely BCN or MSK):", (~has_lesion).sum())

print("\nSample lesion_id values:")
print(meta[has_lesion]['lesion_id'].head(10).tolist())

In [ ]:
import pandas as pd

meta_path = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
meta = pd.read_csv(meta_path)

has_lesion = meta['lesion_id'].notna()
prefixes = meta[has_lesion]['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
print("Lesion ID prefix counts:")
print(prefixes.value_counts())

In [ ]:
import os, pandas as pd

meta_path = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
meta = pd.read_csv(meta_path)

meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]

isic_dedup_ids = meta[meta['prefix'] != 'HAM']['image'].tolist()

print("ISIC images before dedup:", len(meta))
print("ISIC images after removing HAM prefix:", len(isic_dedup_ids))
print("Removed:", len(meta) - len(isic_dedup_ids))

In [ ]:
# ============================================================
# SKIN PRE FLIGHT CHECK. Run once before Stage 0/1.
# Confirms paths, the HAM in ISIC removal, the AK/SCC drop,
# and the final locked numbers all agree.
# ============================================================
import os, pandas as pd

mark = lambda ok: "PASS" if ok else "FAIL"

ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']

print("="*64, "\n1. PATH RESOLUTION\n", "="*64, sep="")
for name, p in [('ISIC_IMG_DIR',ISIC_IMG_DIR), ('ISIC_CSV',ISIC_CSV), ('ISIC_META',ISIC_META),
                ('HAM_IMG_DIR_1',HAM_IMG_DIR_1), ('HAM_IMG_DIR_2',HAM_IMG_DIR_2), ('HAM_CSV',HAM_CSV)]:
    print(f"  [{mark(os.path.exists(p))}] {name}: {p}")

print("\n" + "="*64 + "\n2. HAM REMOVAL FROM ISIC\n" + "="*64)
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
removed = len(meta) - len(isic_keep_ids)
print(f"  ISIC total: {len(meta)} (expect 25331)")
print(f"  [{mark(len(meta)==25331)}] total matches expected")
print(f"  keeping non HAM: {len(isic_keep_ids)} (expect 15316)")
print(f"  [{mark(len(isic_keep_ids)==15316)}] non HAM count matches expected")
print(f"  removed as HAM: {removed} (expect 10015)")
print(f"  [{mark(removed==10015)}] removed count matches expected")

print("\n" + "="*64 + "\n3. ISIC LABEL BUILD, AK/SCC DROP, ALL ZERO GUARD\n" + "="*64)
isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
guard_ok = (row_max == 1).all()
print(f"  [{mark(guard_ok)}] all zero one hot guard (no rows with zero positive label)")
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['exists'] = isic['image_path'].apply(os.path.exists)
isic_before_drop = len(isic)
isic = isic[isic['exists']]
isic = isic[isic['label'].notna()]
print(f"  ISIC rows before AK/SCC drop and file check: {isic_before_drop}")
print(f"  ISIC rows after AK/SCC drop and file check: {len(isic)}")
isic = isic[['image_path','label']].reset_index(drop=True)
isic['source'] = 'isic2019'

print("\n" + "="*64 + "\n4. HAM LABEL BUILD + LESION CHECK\n" + "="*64)
ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham_exists = ham['image_path'].notna().sum()
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
print(f"  HAM images resolved: {ham_exists} / {len(pd.read_csv(HAM_CSV))}")
print(f"  HAM images after label drop: {len(ham)} (expect 9688)")
print(f"  [{mark(len(ham)==9688)}] HAM count matches expected")
print(f"  HAM unique lesions: {ham['lesion_id'].nunique()} (expect materially fewer than images)")
ham_out = ham[['image_path','label','lesion_id']].reset_index(drop=True)
ham_out['source'] = 'ham10000'

print("\n" + "="*64 + "\n5. FINAL POOLED NUMBERS vs LOCKED TABLE\n" + "="*64)
isic['lesion_id'] = None
master_df = pd.concat([isic, ham_out], ignore_index=True)
print(f"  ISIC final: {len(isic)} (expect ~14148)")
print(f"  HAM final: {len(ham_out)} (expect 9688)")
print(f"  Pooled: {len(master_df)} (expect ~23836)")
mel_isic = len(master_df[(master_df.label=='melanoma') & (master_df.source=='isic2019')])
print(f"  ISIC melanoma: {mel_isic} (expect ~3409, NOT 4522, NOT 6017)")
print("\n  Per class per source:")
print(pd.crosstab(master_df['label'], master_df['source']).reindex(FINAL_CLASSES))

print("\n" + "="*64 + "\nDONE. Every PASS/FAIL line above should read PASS.\n" + "="*64)

In [ ]:
import pandas as pd

meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]

non_ham = meta[meta['prefix'] != 'HAM'].copy()
has_lesion = non_ham['lesion_id'].notna()
with_id = non_ham[has_lesion]
without_id = non_ham[~has_lesion]

print("Non-HAM rows total (BCN + MSK + unlinked):", len(non_ham))
print("Non-HAM rows WITH a lesion_id (BCN + MSK):", len(with_id))
print("Non-HAM rows with NO lesion_id at all (truly unlinked):", len(without_id))

counts = with_id['lesion_id'].value_counts()
dup_lesions = counts[counts > 1]

print("\nUnique lesion_ids in BCN + MSK:", with_id['lesion_id'].nunique())
print("Lesion_ids appearing more than once:", len(dup_lesions))
print("Total images sitting inside a duplicated lesion_id:", int(dup_lesions.sum()) if len(dup_lesions) else 0)

if len(dup_lesions):
    dup_rows = with_id[with_id['lesion_id'].isin(dup_lesions.index)]
    print("\nDuplicates broken down by source prefix:")
    print(dup_rows['prefix'].value_counts())
    print("\nSample duplicated lesion_ids and their counts:")
    print(dup_lesions.head(10))

In [ ]:
import os, pandas as pd

ISIC_META = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
print("Exists:", os.path.exists(ISIC_META))

meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]

non_ham = meta[meta['prefix'] != 'HAM'].copy()
has_lesion = non_ham['lesion_id'].notna()
with_id = non_ham[has_lesion]
without_id = non_ham[~has_lesion]

print("Non-HAM rows total (BCN + MSK + unlinked):", len(non_ham))
print("Non-HAM rows WITH a lesion_id (BCN + MSK):", len(with_id))
print("Non-HAM rows with NO lesion_id at all (truly unlinked):", len(without_id))

counts = with_id['lesion_id'].value_counts()
dup_lesions = counts[counts > 1]

print("\nUnique lesion_ids in BCN + MSK:", with_id['lesion_id'].nunique())
print("Lesion_ids appearing more than once:", len(dup_lesions))
print("Total images sitting inside a duplicated lesion_id:", int(dup_lesions.sum()) if len(dup_lesions) else 0)

if len(dup_lesions):
    dup_rows = with_id[with_id['lesion_id'].isin(dup_lesions.index)]
    print("\nDuplicates broken down by source prefix:")
    print(dup_rows['prefix'].value_counts())
    print("\nSample duplicated lesion_ids and their counts:")
    print(dup_lesions.head(10))

In [ ]:
# ===== CELL 0a: SEED + DETERMINISM (must be first executable cell) =====
import os, random
import numpy as np
import tensorflow as tf
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      # GPU op determinism for a defensible seeded run
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
# Optional stronger determinism (TF 2.19): tf.config.experimental.enable_op_determinism()
print(f"Seed {SEED} set, TF {tf.__version__}")   # expect TF 2.19.0

In [ ]:
# ===== CELL 0b: IMPORTS =====
# If the ImageDataGenerator line raises ImportError, Keras 3 removed it on this image.
# Fix: put os.environ['TF_USE_LEGACY_KERAS']='1' at the very top, restart, then `import tf_keras`.
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

In [ ]:
# ---------- PATHS ----------
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
WEIGHTS_DIR   = '/kaggle/input/datasets/ab0y04/all-best-models/'
WORK_DIR      = '/kaggle/working/'

# ---------- CLASSES ----------
FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']   # alphabetical
NUM_CLASSES   = 6
# class_indices (from generator): bcc 0, bkl 1, df 2, melanoma 3, nevus 4, vasc 5

# ---------- IMAGE / BATCH ----------
IMG_SIZE   = 224
BATCH_SIZE = 32

# ---------- SEED ----------
SEED       = 42
CV_SEEDS   = [42, 123, 456, 789, 1024]

# ---------- TRAINING ----------
PHASE1_EPOCHS = 10        # head only, backbone frozen
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5      # full fine-tune
CUSTOM_LR     = 1e-3      # custom CNN, from scratch
EARLYSTOP_PAT = 7         # monitor val_accuracy, restore_best_weights=True
MONITOR       = 'val_accuracy'

# ---------- AUGMENTATION (train only). Defined ONCE. vertical_flip OFF (D1) ----------
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- WEIGHT FILE NAMES ----------
W_CUSTOM = 'best_custom_cnn_c.keras'
W_EFF    = 'best_efficientnet_c.keras'
W_MOB    = 'best_mobilenet_c.keras'
W_RES    = 'best_resnet50_c.keras'

print("Config loaded")

In [ ]:
# ===== CELL 1: BUILD DISJOINT, CLEAN, POOLED DATAFRAME, KEEPING REAL LESION IDS =====
import os

# --- STEP 1: identify non-HAM ISIC images via the metadata lesion-id prefix ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])   # HAM rows excluded, NaN-prefix kept
meta_lesion_map = meta.set_index('image')['lesion_id']            # image -> real lesion_id, NaN where absent
print("ISIC total:", len(meta), "| keeping non-HAM:", len(isic_keep_ids),
      "| removing HAM:", len(meta) - len(isic_keep_ids))          # expect 25331, 15316, 10015

# --- STEP 2: ISIC labels, HAM removed, AK/SCC dropped, with all-zero guard, REAL lesion_id kept ---
isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']          # full one-hot, UNK excluded
# GUARD: no all-zero one-hot rows (an all-zero row would silently become MEL via idxmax)
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label; would mislabel as MEL"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}          # AK/SCC dropped properly
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)             # REAL lesion_id, NOT hardcoded None
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]                                # AK/SCC rows removed
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

# --- STEP 3: HAM, keeps lesion_id for lesion-level split, akiec dropped ---
ham = pd.read_csv(HAM_CSV)                                         # has lesion_id, image_id, dx
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}                  # akiec dropped, keeps 6-class symmetry
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)

# --- STEP 4: pool and verify ---
master_df = pd.concat([isic, ham], ignore_index=True)
print(f"\nISIC (BCN+MSK, AK/SCC dropped) {len(isic):,} | HAM {len(ham):,} | Pooled {len(master_df):,}")
print("\nPer-class per-source:")
print(pd.crosstab(master_df['label'], master_df['source']).reindex(FINAL_CLASSES))
print("\nISIC melanoma (expect 3409, NOT 4522 or 6017):",
      len(master_df[(master_df.label=='melanoma') & (master_df.source=='isic2019')]))

# --- STEP 5: lesion linkage summary. TWO measurement points, do not confuse them ---
print("\nHAM images:", len(ham), "| HAM unique lesions:", ham['lesion_id'].nunique())
# REFERENCE: measured on the RAW non-HAM metadata, BEFORE the AK/SCC drop. Matches the live pre-flight.
ref = meta[meta['prefix'] != 'HAM']
print("REFERENCE (pre-drop, raw metadata): linked", int(ref['lesion_id'].notna().sum()),
      "| unique lesions", ref.loc[ref['lesion_id'].notna(),'lesion_id'].nunique(),
      "| unlinked", int(ref['lesion_id'].isna().sum()))            # expect 13232, 4377, 2084
# ACTUAL: what Stage 2 actually splits on, AFTER the AK/SCC drop and file filter. Will be lower.
isic_linked = isic['lesion_id'].notna().sum()
isic_unlinked = isic['lesion_id'].isna().sum()
print("ACTUAL (post-drop, what Stage 2 splits): linked", int(isic_linked),
      "| unique lesions", isic['lesion_id'].nunique(),
      "| unlinked", int(isic_unlinked), " (lower than reference: ~1168 AK/SCC BCN images removed)")

# --- STEP 6: disjointness guarantee (sources) ---
# ISIC now excludes all HAM-origin images by construction (Step 1). Sources are disjoint.

In [ ]:
# ===== CELL 2a0: PRE-SPLIT DIAGNOSTIC. Look at rare-class lesion counts BEFORE splitting =====
print("HAM unique lesions per class (risk: any class near 1-2 lesions may trigger a fallback):")
print(ham.drop_duplicates('lesion_id')['label'].value_counts())

print("\nISIC linked unique lesions per class (same risk):")
print(isic[isic['lesion_id'].notna()].drop_duplicates('lesion_id')['label'].value_counts())

In [ ]:
# ===== CELL 2a: SAFE SPLIT WRAPPER + LESION-LEVEL SPLIT FOR BOTH ISIC AND HAM =====

def safe_split(df, label_col, test_size, rs, tag=""):
    """Try a stratified split. If any class is too small to stratify, fall back
    to an unstratified split for this step only, and print which group triggered it."""
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, class too small. Falling back to unstratified split.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    # every lesion has one dx, so all its images share a label; stratify lesions by that label
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    """ISIC has two honest categories: rows with a real lesion_id (split at lesion level)
    and rows with none (split at image level, disclosed limitation, no key to group on)."""
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()

    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")

    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")

# leakage gates, BOTH sources
assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and \
       isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and \
       isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LESION LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LESION LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")

train_df = pd.concat([i_tr, h_tr], ignore_index=True)
val_df   = pd.concat([i_va, h_va], ignore_index=True)
test_df  = pd.concat([i_te, h_te], ignore_index=True)

for name, d in [('train',train_df),('val',val_df),('test',test_df)]:
    d.to_csv(f'/kaggle/working/{name}_split_clean.csv', index=False)

print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print("Test per-class:\n", test_df['label'].value_counts().reindex(FINAL_CLASSES))

In [ ]:
# ===== CELL 0a (RESTART FIX): force legacy Keras 2 to avoid the Keras 3 dtype bug =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # must be set before tensorflow is imported

import random
import numpy as np
import tensorflow as tf
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}")
print("tf.keras module:", tf.keras.__name__)   # should now show tf_keras, not keras

In [ ]:
# ===== CELL 0b: IMPORTS =====
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

In [ ]:
# ---------- PATHS ----------
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
WEIGHTS_DIR   = '/kaggle/input/datasets/ab0y04/all-best-models/'
WORK_DIR      = '/kaggle/working/'

# ---------- CLASSES ----------
FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']   # alphabetical
NUM_CLASSES   = 6

# ---------- IMAGE / BATCH ----------
IMG_SIZE   = 224
BATCH_SIZE = 32

# ---------- SEED ----------
SEED       = 42
CV_SEEDS   = [42, 123, 456, 789, 1024]

# ---------- TRAINING ----------
PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5
CUSTOM_LR     = 1e-3
EARLYSTOP_PAT = 7
MONITOR       = 'val_accuracy'

# ---------- AUGMENTATION (train only). Defined ONCE. vertical_flip OFF (D1) ----------
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- WEIGHT FILE NAMES ----------
W_CUSTOM = 'best_custom_cnn_c.keras'
W_EFF    = 'best_efficientnet_c.keras'
W_MOB    = 'best_mobilenet_c.keras'
W_RES    = 'best_resnet50_c.keras'

print("Config loaded")

In [ ]:
# ===== CELL 1: BUILD DISJOINT, CLEAN, POOLED DATAFRAME, KEEPING REAL LESION IDS =====
import os

# --- STEP 1: identify non-HAM ISIC images via the metadata lesion-id prefix ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']
print("ISIC total:", len(meta), "| keeping non-HAM:", len(isic_keep_ids),
      "| removing HAM:", len(meta) - len(isic_keep_ids))

# --- STEP 2: ISIC labels, HAM removed, AK/SCC dropped, with all-zero guard, REAL lesion_id kept ---
isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label; would mislabel as MEL"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

# --- STEP 3: HAM, keeps lesion_id for lesion-level split, akiec dropped ---
ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)

# --- STEP 4: pool and verify ---
master_df = pd.concat([isic, ham], ignore_index=True)
print(f"\nISIC (BCN+MSK, AK/SCC dropped) {len(isic):,} | HAM {len(ham):,} | Pooled {len(master_df):,}")
print("\nPer-class per-source:")
print(pd.crosstab(master_df['label'], master_df['source']).reindex(FINAL_CLASSES))
print("\nISIC melanoma (expect 3409, NOT 4522 or 6017):",
      len(master_df[(master_df.label=='melanoma') & (master_df.source=='isic2019')]))

# --- STEP 5: lesion linkage summary ---
print("\nHAM images:", len(ham), "| HAM unique lesions:", ham['lesion_id'].nunique())
ref = meta[meta['prefix'] != 'HAM']
print("REFERENCE (pre-drop, raw metadata): linked", int(ref['lesion_id'].notna().sum()),
      "| unique lesions", ref.loc[ref['lesion_id'].notna(),'lesion_id'].nunique(),
      "| unlinked", int(ref['lesion_id'].isna().sum()))
isic_linked = isic['lesion_id'].notna().sum()
isic_unlinked = isic['lesion_id'].isna().sum()
print("ACTUAL (post-drop, what Stage 2 splits): linked", int(isic_linked),
      "| unique lesions", isic['lesion_id'].nunique(),
      "| unlinked", int(isic_unlinked), " (lower than reference: ~1168 AK/SCC BCN images removed)")

# --- STEP 6: disjointness guarantee (sources) ---

In [ ]:
# ===== CELL 2a0: PRE-SPLIT DIAGNOSTIC =====
print("HAM unique lesions per class (risk: any class near 1-2 lesions may trigger a fallback):")
print(ham.drop_duplicates('lesion_id')['label'].value_counts())

print("\nISIC linked unique lesions per class (same risk):")
print(isic[isic['lesion_id'].notna()].drop_duplicates('lesion_id')['label'].value_counts())

In [ ]:
# ===== CELL 2a: SAFE SPLIT WRAPPER + LESION-LEVEL SPLIT FOR BOTH ISIC AND HAM =====

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, class too small. Falling back to unstratified split.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()

    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")

    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")

assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and \
       isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and \
       isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LESION LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LESION LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")

train_df = pd.concat([i_tr, h_tr], ignore_index=True)
val_df   = pd.concat([i_va, h_va], ignore_index=True)
test_df  = pd.concat([i_te, h_te], ignore_index=True)

for name, d in [('train',train_df),('val',val_df),('test',test_df)]:
    d.to_csv(f'/kaggle/working/{name}_split_clean.csv', index=False)

print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print("Test per-class:\n", test_df['label'].value_counts().reindex(FINAL_CLASSES))

In [ ]:
# ===== CELL 2b: BALANCED CLASS WEIGHTS (aligned to FINAL_CLASSES order) =====
cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}
print("class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

# ===== FIX-A: bake balanced class weights into a per-row sample_weight column =====
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(train_df[['label','sample_weight']].drop_duplicates().sort_values('label').to_string(index=False))

In [ ]:
# ===== CELL 2c: GENERATOR FACTORY (sample_weight via weight_col, routes around Keras 3 bug) =====
def make_gens(preprocess_fn):
    """preprocess_fn=None -> custom CNN path (rescale 1./255). Else pretrained path (no rescale).
    Train generator yields sample_weight per row via weight_col. Val/test unweighted."""
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label',
                  target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED,
                                        weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df,   shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df,  shuffle=False, **common)
    return tr, va, te

_t,_v,_e = make_gens(None)
print(_e.class_indices)

In [ ]:
# ===== CELL 3a: CUSTOM CNN (3-block, from scratch) =====
def build_custom_cnn(num_classes=6, shape=(224,224,3)):
    m = Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])
    return m

# ===== CELL 3b: PRETRAINED BUILDER (backbone nested -> Grad-CAM friendly) =====
def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'),
        Dropout(0.3),
        Dense(num_classes,activation='softmax')
    ])
    return model, base

print(build_custom_cnn().count_params(), "custom params (record this as new canonical)")

In [ ]:
# ===== TRAINING TEMPLATE, CUSTOM CNN (Stage 4, sample_weight routed via generator) =====
model = build_custom_cnn()
tr, va, te = make_gens(None)                         # rescale 1./255 path, tr yields sample_weight
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/{W_CUSTOM}', monitor='val_accuracy', save_best_only=True),
       CSVLogger('/kaggle/working/custom_log.csv')]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST:", model.evaluate(te, verbose=0))

In [ ]:
# ===== TRAINING TEMPLATE, EfficientNetB0, PHASE 1 ONLY (head, backbone frozen) =====
model, base = build_pretrained(EfficientNetB0)
tr, va, te  = make_gens(eff_pre)

base.trainable = False
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
model.fit(tr, validation_data=va, epochs=10, verbose=1)

# explicit save, Phase 1 checkpoint, separate from the Phase 2 W_EFF checkpoint name
model.save('/kaggle/working/phase1_efficientnet.keras')
print("Phase 1 saved to /kaggle/working/phase1_efficientnet.keras")
print("PHASE 1 sanity check (NOT the final number, backbone still frozen):", model.evaluate(te, verbose=0))

In [ ]:
# ===== FULL RESTART REBUILD: seed/legacy-Keras fix, imports, config, data, split, weights, generators, reload Phase 1 =====

# --- seed + legacy keras fix (must precede TF import) ---
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

# --- imports ---
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

# --- config ---
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
WEIGHTS_DIR   = '/kaggle/input/datasets/ab0y04/all-best-models/'
WORK_DIR      = '/kaggle/working/'

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES   = 6
IMG_SIZE   = 224
BATCH_SIZE = 32
CV_SEEDS   = [42, 123, 456, 789, 1024]
PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5
CUSTOM_LR     = 1e-3
EARLYSTOP_PAT = 7
MONITOR       = 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
W_CUSTOM = 'best_custom_cnn_c.keras'
W_EFF    = 'best_efficientnet_c.keras'
W_MOB    = 'best_mobilenet_c.keras'
W_RES    = 'best_resnet50_c.keras'
print("Config loaded")

# --- Stage 1: data build ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']

isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label; would mislabel as MEL"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)

print(f"ISIC {len(isic):,} | HAM {len(ham):,} | Pooled {len(isic)+len(ham):,}  (expect 14148 / 9688 / 23836)")

# --- Stage 2: split ---
def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, class too small. Falling back to unstratified split.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()
    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))
    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")
    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")

assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and \
       isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and \
       isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LESION LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LESION LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")

train_df = pd.concat([i_tr, h_tr], ignore_index=True)
val_df   = pd.concat([i_va, h_va], ignore_index=True)
test_df  = pd.concat([i_te, h_te], ignore_index=True)
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}  (expect 16752 / 3549 / 3535)")

# --- class weights, sample_weight column ---
cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print("class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

# --- generator factory ---
def make_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label',
                  target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED,
                                        weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df,   shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df,  shuffle=False, **common)
    return tr, va, te

tr, va, te = make_gens(eff_pre)
print(te.class_indices)   # must be {'bcc':0,'bkl':1,'df':2,'melanoma':3,'nevus':4,'vasc':5}


In [ ]:
!find /kaggle/input -iname "phase1_efficientnet.keras" 2>/dev/null

In [ ]:
!find /kaggle/input -iname "best_custom_cnn_c.keras" 2>/dev/null

In [ ]:
# ===== reload Phase 1 EfficientNetB0 model (load_model, never hand-rebuild, Invariant 3) =====
EFF_PHASE1_PATH = '/kaggle/input/datasets/ab0y04/eff-net-head/phase1_efficientnet.keras'

model = load_model(EFF_PHASE1_PATH)
base = model.layers[0]   # nested EfficientNetB0 backbone
print("Phase 1 model reloaded. Base trainable currently:", base.trainable)
print("RE-EVALUATE against locked Phase 1 sanity number (expect close to loss 1.1243, accuracy 0.5437):",
      model.evaluate(te, verbose=0))

In [ ]:
# ===== EfficientNetB0 PHASE 2: unfreeze backbone, recompile, fine-tune =====
base.trainable = True
model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/{W_EFF}', monitor='val_accuracy', save_best_only=True),
       CSVLogger('/kaggle/working/eff_log.csv')]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST:", model.evaluate(te, verbose=0))

In [ ]:
# ===== FULL REBUILD (new session) + STAGE 6: MobileNetV2, BOTH PHASES STRAIGHT THROUGH =====

# --- seed + legacy keras fix (must precede TF import) ---
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

# --- imports ---
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

# --- config ---
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
WEIGHTS_DIR   = '/kaggle/input/datasets/ab0y04/all-best-models/'
WORK_DIR      = '/kaggle/working/'

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES   = 6
IMG_SIZE   = 224
BATCH_SIZE = 32
CV_SEEDS   = [42, 123, 456, 789, 1024]
PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5
CUSTOM_LR     = 1e-3
EARLYSTOP_PAT = 7
MONITOR       = 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
W_CUSTOM = 'best_custom_cnn_c.keras'
W_EFF    = 'best_efficientnet_c.keras'
W_MOB    = 'best_mobilenet_c.keras'
W_RES    = 'best_resnet50_c.keras'
print("Config loaded")

# --- Stage 1: data build ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']

isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label; would mislabel as MEL"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)

print(f"ISIC {len(isic):,} | HAM {len(ham):,} | Pooled {len(isic)+len(ham):,}  (expect 14148 / 9688 / 23836)")

# --- Stage 2: split ---
def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, class too small. Falling back to unstratified split.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()
    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))
    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")
    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")

assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and \
       isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and \
       isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LESION LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LESION LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")

train_df = pd.concat([i_tr, h_tr], ignore_index=True)
val_df   = pd.concat([i_va, h_va], ignore_index=True)
test_df  = pd.concat([i_te, h_te], ignore_index=True)
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}  (expect 16752 / 3549 / 3535)")

# --- class weights, sample_weight column ---
cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print("class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

# --- generator factory ---
def make_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label',
                  target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED,
                                        weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df,   shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df,  shuffle=False, **common)
    return tr, va, te

# --- model builder (Section 8) ---
def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'),
        Dropout(0.3),
        Dense(num_classes,activation='softmax')
    ])
    return model, base

print("\n===== Setup complete. Starting Stage 6: MobileNetV2 =====\n")

# ===== STAGE 6: MobileNetV2, PHASE 1 then PHASE 2, straight through =====
model, base = build_pretrained(MobileNetV2)
tr, va, te  = make_gens(mob_pre)
print(te.class_indices)   # must be {'bcc':0,'bkl':1,'df':2,'melanoma':3,'nevus':4,'vasc':5}

# ---- Phase 1: head only ----
base.trainable = False
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
model.fit(tr, validation_data=va, epochs=10, verbose=1)
print("PHASE 1 sanity check (NOT final, backbone still frozen):", model.evaluate(te, verbose=0))

# ---- Phase 2: full fine-tune (RECOMPILE after unfreeze, Invariant 5) ----
base.trainable = True
model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/{W_MOB}', monitor='val_accuracy', save_best_only=True),
       CSVLogger('/kaggle/working/mob_log.csv')]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST:", model.evaluate(te, verbose=0))

In [ ]:
# ===== STAGE 7: ResNet50, PHASE 1 (now logged) then PHASE 2, straight through =====
model, base = build_pretrained(ResNet50)
tr, va, te  = make_gens(res_pre)
print(te.class_indices)   # must be {'bcc':0,'bkl':1,'df':2,'melanoma':3,'nevus':4,'vasc':5}

# ---- Phase 1: head only (CSVLogger added per new logging rule) ----
base.trainable = False
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
phase1_cbs = [CSVLogger('/kaggle/working/res_phase1_log.csv')]
model.fit(tr, validation_data=va, epochs=10, callbacks=phase1_cbs, verbose=1)
print("PHASE 1 sanity check (NOT final, backbone still frozen):", model.evaluate(te, verbose=0))

In [ ]:
# ===== SAVE PHASE 1 RESNET50 (explicit checkpoint before resuming later) =====
model.save('/kaggle/working/phase1_resnet50.keras')
print("Phase 1 ResNet50 saved to /kaggle/working/phase1_resnet50.keras")

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
# ===== FULL REBUILD (new session) + RELOAD RESNET50 PHASE 1 =====

# --- seed + legacy keras fix (must precede TF import) ---
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

# --- imports ---
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

# --- config ---
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
WORK_DIR      = '/kaggle/working/'

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES   = 6
IMG_SIZE   = 224
BATCH_SIZE = 32
CV_SEEDS   = [42, 123, 456, 789, 1024]
PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5
CUSTOM_LR     = 1e-3
EARLYSTOP_PAT = 7
MONITOR       = 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
W_CUSTOM = 'best_custom_cnn_c.keras'
W_EFF    = 'best_efficientnet_c.keras'
W_MOB    = 'best_mobilenet_c.keras'
W_RES    = 'best_resnet50_c.keras'
print("Config loaded")

# --- Stage 1: data build ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']

isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label; would mislabel as MEL"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)

print(f"ISIC {len(isic):,} | HAM {len(ham):,} | Pooled {len(isic)+len(ham):,}  (expect 14148 / 9688 / 23836)")

# --- Stage 2: split ---
def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, class too small. Falling back to unstratified split.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()
    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))
    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")
    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")

assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and \
       isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and \
       isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LESION LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LESION LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")

train_df = pd.concat([i_tr, h_tr], ignore_index=True)
val_df   = pd.concat([i_va, h_va], ignore_index=True)
test_df  = pd.concat([i_te, h_te], ignore_index=True)
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}  (expect 16752 / 3549 / 3535)")

# --- class weights, sample_weight column ---
cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print("class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

# --- generator factory ---
def make_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label',
                  target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED,
                                        weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df,   shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df,  shuffle=False, **common)
    return tr, va, te

tr, va, te = make_gens(res_pre)
print(te.class_indices)   # must be {'bcc':0,'bkl':1,'df':2,'melanoma':3,'nevus':4,'vasc':5}

# --- reload Phase 1 ResNet50 (load_model, never hand-rebuild, Invariant 3) ---
RES_PHASE1_PATH = '/kaggle/input/datasets/ab0y04/res-net-head/phase1_resnet50.keras'
model = load_model(RES_PHASE1_PATH)
base = model.layers[0]   # nested ResNet50 backbone
print("Phase 1 model reloaded. Base trainable currently:", base.trainable)
print("RE-EVALUATE against locked Phase 1 sanity number (expect close to loss 0.9855, accuracy 0.5895):",
      model.evaluate(te, verbose=0))

In [ ]:
# ===== ResNet50 PHASE 2: unfreeze backbone, recompile, fine-tune =====
base.trainable = True
model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/{W_RES}', monitor='val_accuracy', save_best_only=True),
       CSVLogger('/kaggle/working/res_log.csv')]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST:", model.evaluate(te, verbose=0))

In [ ]:
!find /kaggle/input -iname "best_custom_cnn_c.keras" -o -iname "best_efficientnet_c.keras" -o -iname "best_mobilenet_c.keras" -o -iname "best_resnet50_c.keras" 2>/dev/null

In [ ]:
# ===== STAGE 8: RELOAD ALL FOUR MODELS, EVALUATE UNIFORMLY, LOCK RESULTS TABLE =====
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import gc

MODEL_PATHS = {
    'Custom CNN':     ('/kaggle/input/datasets/ab0y04/best-cnn-v2/best_custom_cnn_c.keras',     None),
    'MobileNetV2':    ('/kaggle/input/datasets/ab0y04/best-mobile-net/best_mobilenet_c.keras',  mob_pre),
    'EfficientNetB0': ('/kaggle/input/datasets/ab0y04/best-eff-net/best_efficientnet_c.keras',  eff_pre),
    'ResNet50':       ('/kaggle/input/datasets/ab0y04/best-res-net/best_resnet50_c.keras',      res_pre),
}

# locked numbers from Stages 4-7, to check reload integrity against
LOCKED_ACC = {'Custom CNN': 0.4973, 'MobileNetV2': 0.7185, 'EfficientNetB0': 0.7429, 'ResNet50': 0.7627}

results = {}
for name, (path, prep_fn) in MODEL_PATHS.items():
    m = load_model(path)
    _, _, te_gen = make_gens(prep_fn)   # own correctly-matched preprocessing, per model
    te_gen.reset()
    y_true = te_gen.classes
    y_prob = m.predict(te_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=FINAL_CLASSES, digits=4)

    # per-class specificity (one-vs-rest, computed from confusion matrix, sklearn has no built-in)
    specificity = {}
    total = cm.sum()
    for i, cls in enumerate(FINAL_CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        specificity[cls] = tn / (tn + fp) if (tn + fp) > 0 else float('nan')

    results[name] = dict(accuracy=acc, macro_f1=macro_f1, cm=cm, report=report,
                          specificity=specificity, y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    print(f"\n===== {name} =====")
    print(f"Re-evaluated accuracy: {acc:.4f}  (locked Stage 4-7 number: {LOCKED_ACC[name]:.4f})")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Confusion matrix (rows=true, cols=pred), order {FINAL_CLASSES}:")
    print(cm)
    print(report)
    print("Specificity per class:", {k: round(v, 4) for k, v in specificity.items()})

    del m
    gc.collect()
    tf.keras.backend.clear_session()   # free GPU memory before next model loads

print("\n===== SUMMARY TABLE =====")
print(f"{'Model':<16}{'Accuracy':<12}{'Macro F1':<12}")
for name, r in results.items():
    print(f"{name:<16}{r['accuracy']:<12.4f}{r['macro_f1']:<12.4f}")

In [ ]:
# ===== SAVE STAGE 8 RESULTS TO CSV + RAW PREDICTIONS FOR STAGE 9 REUSE =====
import pandas as pd
from sklearn.metrics import classification_report

# summary table
summary_df = pd.DataFrame([
    {'model': name, 'accuracy': r['accuracy'], 'macro_f1': r['macro_f1']}
    for name, r in results.items()
])
summary_df.to_csv('/kaggle/working/stage8_summary.csv', index=False)

# per-class metrics
per_class_rows = []
for name, r in results.items():
    rep_dict = classification_report(r['y_true'], r['y_pred'], target_names=FINAL_CLASSES, output_dict=True)
    for cls in FINAL_CLASSES:
        per_class_rows.append({
            'model': name, 'class': cls,
            'precision': rep_dict[cls]['precision'], 'recall': rep_dict[cls]['recall'],
            'f1': rep_dict[cls]['f1-score'], 'support': rep_dict[cls]['support'],
            'specificity': r['specificity'][cls]
        })
pd.DataFrame(per_class_rows).to_csv('/kaggle/working/stage8_per_class_metrics.csv', index=False)

# confusion matrices, long format
cm_rows = []
for name, r in results.items():
    for i, true_cls in enumerate(FINAL_CLASSES):
        for j, pred_cls in enumerate(FINAL_CLASSES):
            cm_rows.append({'model': name, 'true': true_cls, 'pred': pred_cls, 'count': r['cm'][i, j]})
pd.DataFrame(cm_rows).to_csv('/kaggle/working/stage8_confusion_matrices.csv', index=False)

# raw predictions, needed for Stage 9 McNemar/DeLong, avoids re-running predict()
for name, r in results.items():
    fname = name.lower().replace(' ', '_')
    np.savez(f'/kaggle/working/preds_{fname}.npz', y_true=r['y_true'], y_pred=r['y_pred'], y_prob=r['y_prob'])

print("Saved: stage8_summary.csv, stage8_per_class_metrics.csv, stage8_confusion_matrices.csv, preds_*.npz")

In [ ]:
# ===== STAGE 9a: McNEMAR'S TEST, ALL PAIRWISE COMPARISONS =====
import numpy as np
import pandas as pd
from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar

MODEL_NAMES = ['custom_cnn', 'mobilenetv2', 'efficientnetb0', 'resnet50']
DISPLAY_NAMES = {'custom_cnn':'Custom CNN','mobilenetv2':'MobileNetV2',
                  'efficientnetb0':'EfficientNetB0','resnet50':'ResNet50'}

preds = {}
for name in MODEL_NAMES:
    d = np.load(f'/kaggle/working/preds_{name}.npz')
    preds[name] = dict(y_true=d['y_true'], y_pred=d['y_pred'], y_prob=d['y_prob'])

# sanity: y_true must be identical across all four, same test set, same order
base_yt = preds[MODEL_NAMES[0]]['y_true']
for name in MODEL_NAMES[1:]:
    assert np.array_equal(base_yt, preds[name]['y_true']), f"y_true mismatch for {name}, test order not aligned!"
print("y_true alignment check: PASS, identical test order across all four models")

y_true = base_yt
correct = {name: (preds[name]['y_pred'] == y_true) for name in MODEL_NAMES}

pairs = list(combinations(MODEL_NAMES, 2))
rows = []
for a, b in pairs:
    ca, cb = correct[a], correct[b]
    n11 = int(np.sum(ca & cb))    # both correct
    n10 = int(np.sum(ca & ~cb))   # a correct, b wrong
    n01 = int(np.sum(~ca & cb))   # a wrong, b correct
    n00 = int(np.sum(~ca & ~cb))  # both wrong
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=(n10 + n01) < 25, correction=True)
    rows.append({'pair': f"{DISPLAY_NAMES[a]} vs {DISPLAY_NAMES[b]}",
                  f'{a}_only_correct': n10, f'{b}_only_correct': n01,
                  'statistic': result.statistic, 'p_value': result.pvalue})
    print(f"{DISPLAY_NAMES[a]:16} vs {DISPLAY_NAMES[b]:16}  "
          f"{a} only={n10:4d}  {b} only={n01:4d}  stat={result.statistic:.4f}  p={result.pvalue:.6f}")

mcnemar_df = pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)
m = len(mcnemar_df)
mcnemar_df['holm_alpha'] = 0.05 / (m - mcnemar_df.index)
mcnemar_df['significant_holm'] = mcnemar_df['p_value'] < mcnemar_df['holm_alpha']

print("\n===== Holm-Bonferroni corrected results (sorted by p-value) =====")
print(mcnemar_df[['pair','p_value','holm_alpha','significant_holm']].to_string(index=False))

mcnemar_df.to_csv('/kaggle/working/stage9_mcnemar_results.csv', index=False)
print("\nSaved: stage9_mcnemar_results.csv")

In [ ]:
# ===== STAGE 9b: DeLONG'S TEST, PAIRWISE ROC-AUC COMPARISON (one-vs-rest macro AUC) =====
!pip install scikit-posthocs --quiet 2>/dev/null
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from itertools import combinations
from scipy import stats

def delong_roc_variance(y_true_bin, y_prob):
    """DeLong covariance for a single class's AUC. Returns AUC and its variance."""
    order = np.argsort(-y_prob)
    y_true_sorted = y_true_bin[order]
    n1 = np.sum(y_true_sorted == 1)
    n0 = np.sum(y_true_sorted == 0)
    if n1 == 0 or n0 == 0:
        return np.nan, np.nan
    tx = np.zeros(n1); ty = np.zeros(n0)
    pos_scores = y_prob[y_true_bin == 1]
    neg_scores = y_prob[y_true_bin == 0]
    for i, ps in enumerate(pos_scores):
        tx[i] = (np.sum(neg_scores < ps) + 0.5 * np.sum(neg_scores == ps)) / n0
    for j, ns in enumerate(neg_scores):
        ty[j] = (np.sum(pos_scores > ns) + 0.5 * np.sum(pos_scores == ns)) / n1
    auc = np.mean(tx)
    s01 = np.var(tx, ddof=1) / n1
    s10 = np.var(ty, ddof=1) / n0
    return auc, s01 + s10

def delong_test_macro(y_true, prob_a, prob_b, n_classes):
    """Macro-average DeLong test: per-class AUC diff and variance, averaged across classes."""
    diffs, variances = [], []
    for c in range(n_classes):
        yt_bin = (y_true == c).astype(int)
        auc_a, var_a = delong_roc_variance(yt_bin, prob_a[:, c])
        auc_b, var_b = delong_roc_variance(yt_bin, prob_b[:, c])
        if np.isnan(auc_a) or np.isnan(auc_b):
            continue
        diffs.append(auc_a - auc_b)
        variances.append(var_a + var_b)   # conservative, ignores cross-covariance between models
    diffs = np.array(diffs); variances = np.array(variances)
    mean_diff = np.mean(diffs)
    se = np.sqrt(np.mean(variances) / len(variances))
    z = mean_diff / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return mean_diff, z, p

rows = []
for a, b in pairs:   # reuse `pairs` from Stage 9a
    prob_a, prob_b = preds[a]['y_prob'], preds[b]['y_prob']
    auc_a = roc_auc_score(np.eye(NUM_CLASSES)[y_true], prob_a, average='macro', multi_class='ovr')
    auc_b = roc_auc_score(np.eye(NUM_CLASSES)[y_true], prob_b, average='macro', multi_class='ovr')
    mean_diff, z, p = delong_test_macro(y_true, prob_a, prob_b, NUM_CLASSES)
    rows.append({'pair': f"{DISPLAY_NAMES[a]} vs {DISPLAY_NAMES[b]}",
                  f'{a}_auc': auc_a, f'{b}_auc': auc_b,
                  'auc_diff': mean_diff, 'z': z, 'p_value': p})
    print(f"{DISPLAY_NAMES[a]:16} AUC={auc_a:.4f}  vs  {DISPLAY_NAMES[b]:16} AUC={auc_b:.4f}  "
          f"diff={mean_diff:+.4f}  z={z:.4f}  p={p:.6f}")

delong_df = pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)
m = len(delong_df)
delong_df['holm_alpha'] = 0.05 / (m - delong_df.index)
delong_df['significant_holm'] = delong_df['p_value'] < delong_df['holm_alpha']

print("\n===== Holm-Bonferroni corrected DeLong results =====")
print(delong_df[['pair','auc_diff','p_value','holm_alpha','significant_holm']].to_string(index=False))

delong_df.to_csv('/kaggle/working/stage9_delong_results.csv', index=False)
print("\nSaved: stage9_delong_results.csv")

In [ ]:
# ===== STAGE 9c: CORRELATED DeLONG'S TEST (Sun & Xu 2014 fast algorithm) =====
import numpy as np
from scipy import stats

def compute_midrank(x):
    """Midranks, handling ties, needed for the DeLong placement-value computation."""
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2

def fastDeLong(preds_sorted_transposed, label_1_count):
    """
    preds_sorted_transposed: array [n_models, n_samples], columns sorted so all
    positive-label samples come first, then all negative-label samples.
    Returns per-model AUCs and their full covariance matrix (captures correlation
    between models since they're scored on the same samples).
    """
    m = label_1_count
    n = preds_sorted_transposed.shape[1] - m
    k = preds_sorted_transposed.shape[0]
    pos, neg = preds_sorted_transposed[:, :m], preds_sorted_transposed[:, m:]

    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = compute_midrank(pos[r, :])
        ty[r, :] = compute_midrank(neg[r, :])
        tz[r, :] = compute_midrank(preds_sorted_transposed[r, :])

    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_pair_class(y_true_bin, prob_a, prob_b):
    """Correlated DeLong AUC diff test for ONE class, TWO models."""
    order = np.argsort(-y_true_bin, kind='stable')   # positives first
    y_sorted = y_true_bin[order]
    m = int(y_sorted.sum())
    stacked = np.vstack([prob_a[order], prob_b[order]])
    aucs, cov = fastDeLong(stacked, m)
    diff = aucs[0] - aucs[1]
    var_diff = cov[0,0] + cov[1,1] - 2*cov[0,1]   # THE correlation term, missing before
    var_diff = max(var_diff, 1e-12)
    z = diff / np.sqrt(var_diff)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return diff, var_diff, z, p

rows = []
for a, b in pairs:
    prob_a, prob_b = preds[a]['y_prob'], preds[b]['y_prob']
    class_diffs, class_vars = [], []
    for c in range(NUM_CLASSES):
        yt_bin = (y_true == c).astype(float)
        diff, var_diff, _, _ = delong_pair_class(yt_bin, prob_a[:, c], prob_b[:, c])
        class_diffs.append(diff); class_vars.append(var_diff)
    # macro-combine across classes; classes treated as independent, a standard,
    # minor simplification (the within-class model correlation, the part that
    # actually matters, IS captured correctly above)
    mean_diff = np.mean(class_diffs)
    se = np.sqrt(np.sum(class_vars)) / NUM_CLASSES
    z = mean_diff / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    rows.append({'pair': f"{DISPLAY_NAMES[a]} vs {DISPLAY_NAMES[b]}",
                  'auc_diff': mean_diff, 'z': z, 'p_value': p})
    print(f"{DISPLAY_NAMES[a]:16} vs {DISPLAY_NAMES[b]:16}  diff={mean_diff:+.4f}  z={z:.4f}  p={p:.6f}")

delong_corr_df = pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)
m = len(delong_corr_df)
delong_corr_df['holm_alpha'] = 0.05 / (m - delong_corr_df.index)
delong_corr_df['significant_holm'] = delong_corr_df['p_value'] < delong_corr_df['holm_alpha']
print("\n===== Holm-Bonferroni corrected, CORRELATED DeLong results =====")
print(delong_corr_df[['pair','auc_diff','p_value','holm_alpha','significant_holm']].to_string(index=False))

delong_corr_df.to_csv('/kaggle/working/stage9_delong_correlated_results.csv', index=False)
print("\nSaved: stage9_delong_correlated_results.csv")

In [ ]:
# ===== STAGE 10 FIX, SMOKE TEST: Grad-CAM on CPU, one custom + one pretrained =====
test_path = exemplars['melanoma']['path']
test_cidx = exemplars['melanoma']['class_idx']

def heat_report(tag, h):
    print(f"{tag:16} shape={h.shape}  min={h.min():.4f}  max={h.max():.4f}  "
          f"frac>0.01={(h > 0.01).mean():.3f}")
    if h.max() <= 0:
        print(f"  DEGENERATE: {tag} heatmap is all-zero, gradients did not reach the conv layer")

with tf.device('/CPU:0'):
    # --- Custom CNN: Sequential, manual layer-by-layer path ---
    m = load_model(MODEL_PATHS['Custom CNN'][0])
    h_custom = gradcam_custom(m, load_preprocessed(test_path, None), test_cidx)
    heat_report("Custom CNN", h_custom)
    del m; gc.collect(); tf.keras.backend.clear_session()

    # --- EfficientNetB0: nested backbone, single-pass dual-output path ---
    m = load_model(MODEL_PATHS['EfficientNetB0'][0])
    h_eff = gradcam_pretrained(m, load_preprocessed(test_path, eff_pre), 'top_conv', test_cidx)
    heat_report("EfficientNetB0", h_eff)
    del m; gc.collect(); tf.keras.backend.clear_session()

print("\nSmoke test complete. Both paths ran without error if you see two heat_report lines above.")

In [ ]:
# ===== STAGE 10, FULL GRAD-CAM GRID (CPU, all 4 models x 3 exemplar classes) =====
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
row_labels = ['melanoma', 'bcc', 'df']

with tf.device('/CPU:0'):
    for row, cls_name in enumerate(row_labels):
        ex = exemplars[cls_name]
        raw = load_raw(ex['path'])
        axes[row, 0].imshow(raw)
        axes[row, 0].set_title(f"{cls_name}\n(original)" if row == 0 else cls_name)
        axes[row, 0].axis('off')

        for col, name in enumerate(MODEL_NAMES, start=1):
            disp = DISPLAY_NAMES[name]
            path, _ = MODEL_PATHS[disp]
            m = load_model(path)
            img_arr = load_preprocessed(ex['path'], PREP_FN[disp])

            if disp == 'Custom CNN':
                heat = gradcam_custom(m, img_arr, ex['class_idx'])
            else:
                heat = gradcam_pretrained(m, img_arr, LAST_CONV[disp], ex['class_idx'])

            overlay = overlay_heatmap(raw, heat)
            axes[row, col].imshow(overlay)
            if row == 0:
                axes[row, col].set_title(disp)
            axes[row, col].axis('off')

            del m; gc.collect(); tf.keras.backend.clear_session()

plt.tight_layout()
plt.savefig('/kaggle/working/stage10_gradcam_exemplars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: stage10_gradcam_exemplars.png")

In [ ]:
# ===== STAGE 10, FULL: 4-PANEL CONFUSION MATRIX + GRAD-CAM (melanoma, bcc, df) =====
import matplotlib.pyplot as plt
import matplotlib.cm as cm_colormap
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D
from sklearn.metrics import confusion_matrix
import gc

LAST_CONV = {'Custom CNN': None, 'MobileNetV2': 'Conv_1',
             'EfficientNetB0': 'top_conv', 'ResNet50': 'conv5_block3_out'}
PREP_FN = {'Custom CNN': None, 'MobileNetV2': mob_pre, 'EfficientNetB0': eff_pre, 'ResNet50': res_pre}
CLASS_IDX = {c: i for i, c in enumerate(FINAL_CLASSES)}

# ---------- PART A: 4-PANEL NORMALIZED CONFUSION MATRIX ----------
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, name in zip(axes.flat, MODEL_NAMES):
    yt, yp = preds[name]['y_true'], preds[name]['y_pred']
    cm = confusion_matrix(yt, yp)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels(FINAL_CLASSES, rotation=45, ha='right')
    ax.set_yticklabels(FINAL_CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(DISPLAY_NAMES[name], pad=10)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{cm_norm[i,j]:.2f}", ha='center', va='center',
                     color='white' if cm_norm[i,j] > 0.5 else 'black', fontsize=8)
fig.subplots_adjust(right=0.85, hspace=0.4, wspace=0.3)
cbar_ax = fig.add_axes([0.88, 0.15, 0.03, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Row-normalized proportion')
plt.savefig('/kaggle/working/stage10_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: stage10_confusion_matrices.png")

# ---------- PART B: SELECT EXEMPLARS (melanoma, bcc, df), correctly classified by EfficientNetB0 ----------
eff_probs = preds['efficientnetb0']['y_prob']
eff_pred  = preds['efficientnetb0']['y_pred']
test_df_reset = test_df.reset_index(drop=True)

exemplars = {}
for cls_name in ['melanoma', 'bcc', 'df']:
    c = CLASS_IDX[cls_name]
    mask = (y_true == c) & (eff_pred == c)
    if not mask.any():
        print(f"WARNING: no correctly-classified {cls_name} example from EfficientNetB0, using highest-confidence true example instead")
        mask = (y_true == c)
    candidate_idxs = np.where(mask)[0]
    best_idx = candidate_idxs[np.argmax(eff_probs[candidate_idxs, c])]
    exemplars[cls_name] = dict(row_idx=best_idx, path=test_df_reset.loc[best_idx, 'image_path'], class_idx=c)
    print(f"{cls_name}: row {best_idx}, EfficientNetB0 confidence {eff_probs[best_idx, c]:.4f}")

# ---------- PART C: GRAD-CAM HELPERS ----------
def load_raw(path):
    img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    return img_to_array(img).astype('uint8')

def load_preprocessed(path, prep_fn):
    img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(img_to_array(img), axis=0)
    return (arr / 255.0) if prep_fn is None else prep_fn(arr)

def gradcam_pretrained(model, img_array, last_conv_name, class_idx):
    """Section 8's locked method: dual-output model from base, head applied inside the SAME tape."""
    base = model.layers[0]
    grad_model = Model(inputs=base.input, outputs=[base.get_layer(last_conv_name).output, base.output])
    with tf.GradientTape() as tape:
        conv_out, base_out = grad_model(img_array, training=False)
        x = base_out
        for layer in model.layers[1:]:
            x = layer(x, training=False)
        class_score = x[:, class_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heat = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def gradcam_custom(model, img_array, class_idx):
    """Custom CNN, no nested backbone: manual layer-by-layer forward pass, last Conv2D by index."""
    conv_idxs = [i for i, l in enumerate(model.layers) if isinstance(l, Conv2D)]
    last_idx = conv_idxs[-1]
    x = tf.convert_to_tensor(img_array)
    with tf.GradientTape() as tape:
        tape.watch(x)
        conv_out = None
        for i, layer in enumerate(model.layers):
            x = layer(x, training=False)
            if i == last_idx:
                conv_out = x
        class_score = x[:, class_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heat = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def overlay_heatmap(raw_img, heatmap, alpha=0.4):
    heat_resized = tf.image.resize(heatmap[..., np.newaxis], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    colored = cm_colormap.jet(heat_resized)[:, :, :3] * 255
    return (raw_img * (1 - alpha) + colored * alpha).astype('uint8')

# ---------- PART D: FULL GRAD-CAM GRID, CPU-scoped (GPU has no deterministic BatchNorm-backward kernel) ----------
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
row_labels = ['melanoma', 'bcc', 'df']

with tf.device('/CPU:0'):
    for row, cls_name in enumerate(row_labels):
        ex = exemplars[cls_name]
        raw = load_raw(ex['path'])
        axes[row, 0].imshow(raw)
        axes[row, 0].set_title(f"{cls_name}\n(original)" if row == 0 else cls_name)
        axes[row, 0].axis('off')

        for col, name in enumerate(MODEL_NAMES, start=1):
            disp = DISPLAY_NAMES[name]
            path, _ = MODEL_PATHS[disp]
            m = load_model(path)
            img_arr = load_preprocessed(ex['path'], PREP_FN[disp])

            if disp == 'Custom CNN':
                heat = gradcam_custom(m, img_arr, ex['class_idx'])
            else:
                heat = gradcam_pretrained(m, img_arr, LAST_CONV[disp], ex['class_idx'])

            overlay = overlay_heatmap(raw, heat)
            axes[row, col].imshow(overlay)
            if row == 0:
                axes[row, col].set_title(disp)
            axes[row, col].axis('off')

            del m; gc.collect(); tf.keras.backend.clear_session()

plt.tight_layout()
plt.savefig('/kaggle/working/stage10_gradcam_exemplars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: stage10_gradcam_exemplars.png")

In [ ]:
# ===== FULL REBUILD (new session) + STAGE 8 RE-RUN (pooled + per-source retrofit) =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              f1_score, roc_auc_score)
print("Imports ready")

# --- config ---
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES   = 6
IMG_SIZE   = 224
BATCH_SIZE = 32
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print("Config loaded")

# --- Stage 1: data build ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']

isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)
print(f"ISIC {len(isic):,} | HAM {len(ham):,} | Pooled {len(isic)+len(ham):,}  (expect 14148 / 9688 / 23836)")

# --- Stage 2: split ---
def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()
    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))
    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")
    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")
assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")

train_df = pd.concat([i_tr, h_tr], ignore_index=True)
val_df   = pd.concat([i_va, h_va], ignore_index=True)
test_df  = pd.concat([i_te, h_te], ignore_index=True)
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}  (expect 16752 / 3549 / 3535)")

cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)

def make_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

print("\n===== Setup complete. Re-running Stage 8 =====\n")

# ===== STAGE 8: RELOAD ALL FOUR, EVALUATE, POOLED + PER-SOURCE =====
MODEL_PATHS = {
    'Custom CNN':     ('/kaggle/input/datasets/ab0y04/best-cnn-v2/best_custom_cnn_c.keras',     None),
    'MobileNetV2':    ('/kaggle/input/datasets/ab0y04/best-mobile-net/best_mobilenet_c.keras',  mob_pre),
    'EfficientNetB0': ('/kaggle/input/datasets/ab0y04/best-eff-net/best_efficientnet_c.keras',  eff_pre),
    'ResNet50':       ('/kaggle/input/datasets/ab0y04/best-res-net/best_resnet50_c.keras',      res_pre),
}
MODEL_NAMES = ['custom_cnn', 'mobilenetv2', 'efficientnetb0', 'resnet50']
DISPLAY_NAMES = {'custom_cnn':'Custom CNN','mobilenetv2':'MobileNetV2',
                  'efficientnetb0':'EfficientNetB0','resnet50':'ResNet50'}
LOCKED_ACC = {'Custom CNN': 0.4973, 'MobileNetV2': 0.7185, 'EfficientNetB0': 0.7429, 'ResNet50': 0.7627}

results = {}
preds = {}
for name in MODEL_NAMES:
    disp = DISPLAY_NAMES[name]
    path, prep_fn = MODEL_PATHS[disp]
    m = load_model(path)
    _, _, te_gen = make_gens(prep_fn)
    te_gen.reset()
    y_true = te_gen.classes
    y_prob = m.predict(te_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)

    preds[name] = dict(y_true=y_true, y_pred=y_pred, y_prob=y_prob)
    results[name] = dict(accuracy=acc, macro_f1=macro_f1, cm=cm)
    print(f"{disp:16} re-evaluated accuracy: {acc:.4f}  (locked: {LOCKED_ACC[disp]:.4f})  macro_f1: {macro_f1:.4f}")

    np.savez(f'/kaggle/working/preds_{name}.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)
    del m; gc.collect(); tf.keras.backend.clear_session()

y_true = preds['custom_cnn']['y_true']   # shared across all four, order identical

# ---------- per-source retrofit ----------
test_df_reset = test_df.reset_index(drop=True)
source_arr = test_df_reset['source'].values

rows = []
for name in MODEL_NAMES:
    yt, yp, yp_prob = preds[name]['y_true'], preds[name]['y_pred'], preds[name]['y_prob']
    for src in ['isic2019', 'ham10000']:
        mask = source_arr == src
        acc = accuracy_score(yt[mask], yp[mask])
        f1  = f1_score(yt[mask], yp[mask], average='macro')
        auc = roc_auc_score(np.eye(NUM_CLASSES)[yt[mask]], yp_prob[mask], average='macro', multi_class='ovr')
        rows.append({'model': DISPLAY_NAMES[name], 'source': src, 'n': int(mask.sum()),
                      'accuracy': acc, 'macro_f1': f1, 'macro_auc': auc})

per_source_df = pd.DataFrame(rows)
per_source_df.to_csv('/kaggle/working/stage8_per_source.csv', index=False)
print("\n===== STAGE 8 PER-SOURCE BREAKDOWN =====")
print(per_source_df.to_string(index=False))
print("\nSaved: preds_*.npz (4 files), stage8_per_source.csv")

In [ ]:
# ===== PER-SOURCE RETROFIT, FIXED: explicit dtype casting + integer indices instead of raw boolean mask =====
test_df_reset = test_df.reset_index(drop=True)
source_arr = np.asarray(test_df_reset['source'].values)
print("source_arr dtype:", source_arr.dtype, "shape:", source_arr.shape, "unique:", np.unique(source_arr))
print("y_true (custom_cnn) dtype:", preds['custom_cnn']['y_true'].dtype, "shape:", preds['custom_cnn']['y_true'].shape)

rows = []
for name in MODEL_NAMES:
    yt = np.asarray(preds[name]['y_true'])
    yp = np.asarray(preds[name]['y_pred'])
    yp_prob = np.asarray(preds[name]['y_prob'])
    for src in ['isic2019', 'ham10000']:
        mask = np.asarray(source_arr == src, dtype=bool)
        idx = np.where(mask)[0]   # explicit integer indices, sidesteps whatever the mask dtype issue was
        acc = accuracy_score(yt[idx], yp[idx])
        f1  = f1_score(yt[idx], yp[idx], average='macro')
        auc = roc_auc_score(np.eye(NUM_CLASSES)[yt[idx]], yp_prob[idx], average='macro', multi_class='ovr')
        rows.append({'model': DISPLAY_NAMES[name], 'source': src, 'n': int(len(idx)),
                      'accuracy': acc, 'macro_f1': f1, 'macro_auc': auc})

per_source_df = pd.DataFrame(rows)
per_source_df.to_csv('/kaggle/working/stage8_per_source.csv', index=False)
print("\n===== STAGE 8 PER-SOURCE BREAKDOWN =====")
print(per_source_df.to_string(index=False))
print("\nSaved: stage8_per_source.csv")

In [ ]:
# ===== PER-SOURCE RETROFIT (diagnostic line fixed to convert before reading .dtype) =====
test_df_reset = test_df.reset_index(drop=True)
source_arr = np.asarray(test_df_reset['source'].values)
yt_check = np.asarray(preds['custom_cnn']['y_true'])
print("source_arr dtype:", source_arr.dtype, "shape:", source_arr.shape, "unique:", np.unique(source_arr))
print("y_true (custom_cnn) dtype:", yt_check.dtype, "shape:", yt_check.shape)
print("original storage type was:", type(preds['custom_cnn']['y_true']), "-> confirms te_gen.classes returns a list in tf_keras")

rows = []
for name in MODEL_NAMES:
    yt = np.asarray(preds[name]['y_true'])
    yp = np.asarray(preds[name]['y_pred'])
    yp_prob = np.asarray(preds[name]['y_prob'])
    for src in ['isic2019', 'ham10000']:
        mask = np.asarray(source_arr == src, dtype=bool)
        idx = np.where(mask)[0]
        acc = accuracy_score(yt[idx], yp[idx])
        f1  = f1_score(yt[idx], yp[idx], average='macro')
        auc = roc_auc_score(np.eye(NUM_CLASSES)[yt[idx]], yp_prob[idx], average='macro', multi_class='ovr')
        rows.append({'model': DISPLAY_NAMES[name], 'source': src, 'n': int(len(idx)),
                      'accuracy': acc, 'macro_f1': f1, 'macro_auc': auc})

per_source_df = pd.DataFrame(rows)
per_source_df.to_csv('/kaggle/working/stage8_per_source.csv', index=False)
print("\n===== STAGE 8 PER-SOURCE BREAKDOWN =====")
print(per_source_df.to_string(index=False))
print("\nSaved: stage8_per_source.csv")

In [ ]:
# ===== CELL B: GRAD-CAM REGEN, per-class figures WITH prediction labels =====
import matplotlib.pyplot as plt
import matplotlib.cm as cm_colormap
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model

LAST_CONV = {'Custom CNN': None, 'MobileNetV2': 'Conv_1',
             'EfficientNetB0': 'top_conv', 'ResNet50': 'conv5_block3_out'}
PREP_FN = {'Custom CNN': None, 'MobileNetV2': mob_pre, 'EfficientNetB0': eff_pre, 'ResNet50': res_pre}
CLASS_IDX = {c: i for i, c in enumerate(FINAL_CLASSES)}

eff_probs = preds['efficientnetb0']['y_prob']
eff_pred  = preds['efficientnetb0']['y_pred']

exemplars = {}
for cls_name in ['melanoma', 'bcc', 'df']:
    c = CLASS_IDX[cls_name]
    mask = (y_true == c) & (eff_pred == c)
    if not mask.any():
        print(f"WARNING: no correctly-classified {cls_name} from EfficientNetB0, using best-confidence true example")
        mask = (y_true == c)
    candidate_idxs = np.where(mask)[0]
    best_idx = candidate_idxs[np.argmax(eff_probs[candidate_idxs, c])]
    exemplars[cls_name] = dict(row_idx=best_idx, path=test_df_reset.loc[best_idx, 'image_path'], class_idx=c)
    print(f"{cls_name}: row {best_idx}, path {exemplars[cls_name]['path']}")

def load_raw(path):
    img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    return img_to_array(img).astype('uint8')

def load_preprocessed(path, prep_fn):
    img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(img_to_array(img), axis=0)
    return (arr / 255.0) if prep_fn is None else prep_fn(arr)

def gradcam_pretrained(model, img_array, last_conv_name, class_idx):
    base = model.layers[0]
    grad_model = Model(inputs=base.input, outputs=[base.get_layer(last_conv_name).output, base.output])
    with tf.GradientTape() as tape:
        conv_out, base_out = grad_model(img_array, training=False)
        x = base_out
        for layer in model.layers[1:]:
            x = layer(x, training=False)
        class_score = x[:, class_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heat = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def gradcam_custom(model, img_array, class_idx):
    conv_idxs = [i for i, l in enumerate(model.layers) if isinstance(l, Conv2D)]
    last_idx = conv_idxs[-1]
    x = tf.convert_to_tensor(img_array)
    with tf.GradientTape() as tape:
        tape.watch(x)
        conv_out = None
        for i, layer in enumerate(model.layers):
            x = layer(x, training=False)
            if i == last_idx:
                conv_out = x
        class_score = x[:, class_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heat = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def overlay_heatmap(raw_img, heatmap, alpha=0.4):
    heat_resized = tf.image.resize(heatmap[..., np.newaxis], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    colored = cm_colormap.jet(heat_resized)[:, :, :3] * 255
    return (raw_img * (1 - alpha) + colored * alpha).astype('uint8')

with tf.device('/CPU:0'):
    for cls_name in ['melanoma', 'bcc', 'df']:
        ex = exemplars[cls_name]
        raw = load_raw(ex['path'])
        row_idx = ex['row_idx']

        fig, axes = plt.subplots(1, 5, figsize=(22, 5))
        fig.suptitle(f"Grad-CAM Comparison: {cls_name.upper()}", fontsize=15, fontweight='bold', y=1.02)
        axes[0].imshow(raw)
        axes[0].set_title(f"Original\nTrue: {cls_name}", fontsize=11)
        axes[0].axis('off')

        for col, name in enumerate(MODEL_NAMES, start=1):
            disp = DISPLAY_NAMES[name]
            path, _ = MODEL_PATHS[disp]
            m = load_model(path)
            img_arr = load_preprocessed(ex['path'], PREP_FN[disp])
            pred_class_idx = preds[name]['y_pred'][row_idx]
            pred_label = FINAL_CLASSES[pred_class_idx]

            if disp == 'Custom CNN':
                heat = gradcam_custom(m, img_arr, ex['class_idx'])
            else:
                heat = gradcam_pretrained(m, img_arr, LAST_CONV[disp], ex['class_idx'])
            overlay = overlay_heatmap(raw, heat)

            correct = (pred_label == cls_name)
            axes[col].imshow(overlay)
            axes[col].set_title(f"{disp}\nPred: {pred_label}", fontsize=11,
                                 color='black' if correct else 'firebrick')
            axes[col].axis('off')
            del m; gc.collect(); tf.keras.backend.clear_session()

        plt.tight_layout()
        fname = f'/kaggle/working/gradcam_{cls_name}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: {fname}")

In [ ]:
# ===== CELL B: GRAD-CAM REGEN, per-class figures WITH prediction labels (fixed: y_true cast to array) =====
import matplotlib.pyplot as plt
import matplotlib.cm as cm_colormap
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D

LAST_CONV = {'Custom CNN': None, 'MobileNetV2': 'Conv_1',
             'EfficientNetB0': 'top_conv', 'ResNet50': 'conv5_block3_out'}
PREP_FN = {'Custom CNN': None, 'MobileNetV2': mob_pre, 'EfficientNetB0': eff_pre, 'ResNet50': res_pre}
CLASS_IDX = {c: i for i, c in enumerate(FINAL_CLASSES)}

# --- fix: te_gen.classes returns a plain list under legacy tf_keras, breaks elementwise comparison ---
y_true = np.asarray(preds['custom_cnn']['y_true'])
print("y_true fixed, dtype:", y_true.dtype, "shape:", y_true.shape)

eff_probs = preds['efficientnetb0']['y_prob']
eff_pred  = np.asarray(preds['efficientnetb0']['y_pred'])

# --- exemplar selection ---
exemplars = {}
for cls_name in ['melanoma', 'bcc', 'df']:
    c = CLASS_IDX[cls_name]
    mask = (y_true == c) & (eff_pred == c)
    if not mask.any():
        print(f"WARNING: no correctly-classified {cls_name} from EfficientNetB0, using best-confidence true example")
        mask = (y_true == c)
    candidate_idxs = np.where(mask)[0]
    best_idx = candidate_idxs[np.argmax(eff_probs[candidate_idxs, c])]
    exemplars[cls_name] = dict(row_idx=best_idx, path=test_df_reset.loc[best_idx, 'image_path'], class_idx=c)
    print(f"{cls_name}: row {best_idx}, path {exemplars[cls_name]['path']}")

# --- helpers ---
def load_raw(path):
    img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    return img_to_array(img).astype('uint8')

def load_preprocessed(path, prep_fn):
    img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(img_to_array(img), axis=0)
    return (arr / 255.0) if prep_fn is None else prep_fn(arr)

def gradcam_pretrained(model, img_array, last_conv_name, class_idx):
    base = model.layers[0]
    grad_model = Model(inputs=base.input, outputs=[base.get_layer(last_conv_name).output, base.output])
    with tf.GradientTape() as tape:
        conv_out, base_out = grad_model(img_array, training=False)
        x = base_out
        for layer in model.layers[1:]:
            x = layer(x, training=False)
        class_score = x[:, class_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heat = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def gradcam_custom(model, img_array, class_idx):
    conv_idxs = [i for i, l in enumerate(model.layers) if isinstance(l, Conv2D)]
    last_idx = conv_idxs[-1]
    x = tf.convert_to_tensor(img_array)
    with tf.GradientTape() as tape:
        tape.watch(x)
        conv_out = None
        for i, layer in enumerate(model.layers):
            x = layer(x, training=False)
            if i == last_idx:
                conv_out = x
        class_score = x[:, class_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heat = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def overlay_heatmap(raw_img, heatmap, alpha=0.4):
    heat_resized = tf.image.resize(heatmap[..., np.newaxis], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    colored = cm_colormap.jet(heat_resized)[:, :, :3] * 255
    return (raw_img * (1 - alpha) + colored * alpha).astype('uint8')

# --- generate the three labeled figures ---
with tf.device('/CPU:0'):
    for cls_name in ['melanoma', 'bcc', 'df']:
        ex = exemplars[cls_name]
        raw = load_raw(ex['path'])
        row_idx = ex['row_idx']

        fig, axes = plt.subplots(1, 5, figsize=(22, 5))
        fig.suptitle(f"Grad-CAM Comparison: {cls_name.upper()}", fontsize=15, fontweight='bold', y=1.02)
        axes[0].imshow(raw)
        axes[0].set_title(f"Original\nTrue: {cls_name}", fontsize=11)
        axes[0].axis('off')

        for col, name in enumerate(MODEL_NAMES, start=1):
            disp = DISPLAY_NAMES[name]
            path, _ = MODEL_PATHS[disp]
            m = load_model(path)
            img_arr = load_preprocessed(ex['path'], PREP_FN[disp])
            pred_class_idx = np.asarray(preds[name]['y_pred'])[row_idx]
            pred_label = FINAL_CLASSES[pred_class_idx]

            if disp == 'Custom CNN':
                heat = gradcam_custom(m, img_arr, ex['class_idx'])
            else:
                heat = gradcam_pretrained(m, img_arr, LAST_CONV[disp], ex['class_idx'])
            overlay = overlay_heatmap(raw, heat)

            correct = (pred_label == cls_name)
            axes[col].imshow(overlay)
            axes[col].set_title(f"{disp}\nPred: {pred_label}", fontsize=11,
                                 color='black' if correct else 'firebrick')
            axes[col].axis('off')
            del m; gc.collect(); tf.keras.backend.clear_session()

        plt.tight_layout()
        fname = f'/kaggle/working/gradcam_{cls_name}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: {fname}")

In [6]:
# ===== STAGE 11 SETUP: verify Stage 2's per-source splits are intact, build class weights + generators =====
print("Checking i_tr/i_va/i_te (ISIC) and h_tr/h_va/h_te (HAM) are still in memory...")
for name, df in [('i_tr',i_tr),('i_va',i_va),('i_te',i_te),('h_tr',h_tr),('h_va',h_va),('h_te',h_te)]:
    print(f"  {name}: {len(df):,} rows")

# gate: these must match Stage 8's per-source retrofit test counts exactly, since that table
# was computed FROM these same splits via the pooled concat
assert len(i_te) == 2091, f"ISIC test count drifted: {len(i_te)} vs expected 2091"
assert len(h_te) == 1444, f"HAM test count drifted: {len(h_te)} vs expected 1444"
print("\nGate PASS: single-source test counts match Stage 8's per-source table exactly.")

# --- per-source class weights, computed independently per source, NOT reused from pooled ---
def add_sample_weights(train_src):
    cls = np.array(FINAL_CLASSES)
    cw = compute_class_weight('balanced', classes=cls, y=train_src['label'])
    w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
    train_src = train_src.copy()
    train_src['sample_weight'] = train_src['label'].map(w_map)
    print("  class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})
    return train_src

print("\nISIC-only class weights:")
i_tr_w = add_sample_weights(i_tr)
print("HAM-only class weights:")
h_tr_w = add_sample_weights(h_tr)

# --- generic single-source generator factory ---
def make_source_gens(train_src, val_src, test_src, preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_src, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_src, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_src, shuffle=False, **common)
    return tr, va, te

# --- persist per-source splits (chest's pattern, D-skin-2 spirit: don't lose intermediate state) ---
i_tr_w.to_csv('/kaggle/working/isic_train_split.csv', index=False)
i_va.to_csv('/kaggle/working/isic_val_split.csv', index=False)
i_te.to_csv('/kaggle/working/isic_test_split.csv', index=False)
h_tr_w.to_csv('/kaggle/working/ham_train_split.csv', index=False)
h_va.to_csv('/kaggle/working/ham_val_split.csv', index=False)
h_te.to_csv('/kaggle/working/ham_test_split.csv', index=False)
print("\nSaved: isic_{train,val,test}_split.csv, ham_{train,val,test}_split.csv")

SS_ARCH = {
    'custom': (None, None),
    'mob':    (MobileNetV2, mob_pre),
    'eff':    (EfficientNetB0, eff_pre),
    'res':    (ResNet50, res_pre),
}
print("\n8 runs planned: custom/mob/eff/res x isic/ham")
print("Filenames: ss_{arch}_{isic|ham}.keras")

Checking i_tr/i_va/i_te (ISIC) and h_tr/h_va/h_te (HAM) are still in memory...
  i_tr: 9,963 rows
  i_va: 2,094 rows
  i_te: 2,091 rows
  h_tr: 6,789 rows
  h_va: 1,455 rows
  h_te: 1,444 rows

Gate PASS: single-source test counts match Stage 8's per-source table exactly.

ISIC-only class weights:
  class_weight: {np.str_('bcc'): np.float64(0.852), np.str_('bkl'): np.float64(1.522), np.str_('df'): np.float64(18.049), np.str_('melanoma'): np.float64(0.677), np.str_('nevus'): np.float64(0.386), np.str_('vasc'): np.float64(22.747)}
HAM-only class weights:
  class_weight: {np.str_('bcc'): np.float64(3.017), np.str_('bkl'): np.float64(1.466), np.str_('df'): np.float64(13.969), np.str_('melanoma'): np.float64(1.477), np.str_('nevus'): np.float64(0.241), np.str_('vasc'): np.float64(11.093)}

Saved: isic_{train,val,test}_split.csv, ham_{train,val,test}_split.csv

8 runs planned: custom/mob/eff/res x isic/ham
Filenames: ss_{arch}_{isic|ham}.keras


In [1]:
# ===== FULL REBUILD (new session) + STAGE 11 SETUP =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

# --- config ---
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES   = 6
IMG_SIZE   = 224
BATCH_SIZE = 32
PHASE1_EPOCHS = 10
CUSTOM_LR     = 1e-3
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print("Config loaded")

# --- Stage 1: data build ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']

isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)
print(f"ISIC {len(isic):,} | HAM {len(ham):,} | Pooled {len(isic)+len(ham):,}  (expect 14148 / 9688 / 23836)")

# --- Stage 2: per-source splits (i_tr/i_va/i_te, h_tr/h_va/h_te ARE Stage 11's inputs) ---
def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()
    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))
    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")
    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")
assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")
print(f"i_tr {len(i_tr):,} | i_va {len(i_va):,} | i_te {len(i_te):,}  (expect ~9963 / 2094 / 2091)")
print(f"h_tr {len(h_tr):,} | h_va {len(h_va):,} | h_te {len(h_te):,}  (expect ~6789 / 1455 / 1444)")

# --- model builders (needed for Stage 11 training) ---
def build_custom_cnn(num_classes=6, shape=(224,224,3)):
    m = Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])
    return m

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

# --- Stage 11 setup: per-source class weights, generic generator factory ---
def add_sample_weights(train_src):
    cls = np.array(FINAL_CLASSES)
    cw = compute_class_weight('balanced', classes=cls, y=train_src['label'])
    w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
    train_src = train_src.copy()
    train_src['sample_weight'] = train_src['label'].map(w_map)
    print("  class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})
    return train_src

print("\nISIC-only class weights:")
i_tr_w = add_sample_weights(i_tr)
print("HAM-only class weights:")
h_tr_w = add_sample_weights(h_tr)

def make_source_gens(train_src, val_src, test_src, preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_src, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_src, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_src, shuffle=False, **common)
    return tr, va, te

print("\n===== Rebuild + Stage 11 setup complete. 8 runs planned: custom/mob/eff/res x isic/ham =====")

2026-07-22 10:26:27.886164: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784715988.063710      59 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784715988.117845      59 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784715988.555805      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784715988.555854      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784715988.555858      59 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded
ISIC 14,148 | HAM 9,688 | Pooled 23,836  (expect 14148 / 9688 / 23836)
ISIC lesion-leakage check: PASS
HAM lesion-leakage check: PASS
i_tr 9,963 | i_va 2,094 | i_te 2,091  (expect ~9963 / 2094 / 2091)
h_tr 6,789 | h_va 1,455 | h_te 1,444  (expect ~6789 / 1455 / 1444)

ISIC-only class weights:
  class_weight: {np.str_('bcc'): np.float64(0.852), np.str_('bkl'): np.float64(1.522), np.str_('df'): np.float64(18.049), np.str_('melanoma'): np.float64(0.677), np.str_('nevus'): np.float64(0.386), np.str_('vasc'): np.float64(22.747)}
HAM-only class weights:
  class_weight: {np.str_('bcc'): np.float64(3.017), np.str_('bkl'): np.float64(1.466), np.str_('df'): np.float64(13.969), np.str_('melanoma'): np.float64(1.477), np.str_('nevus'): np.float64(0.241), np.str_('vasc'): np.float64(11.093)}

===== Rebuild + Stage 11 setup complete. 8 runs planned: custom/mob/eff/res x isic/ham =====


In [2]:
# ===== STAGE 11, RUN 1/8: Custom CNN on ISIC-only =====
tr, va, te = make_source_gens(i_tr_w, i_va, i_te, None)
print(te.class_indices)

model = build_custom_cnn()
model.compile(Adam(CUSTOM_LR), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint('/kaggle/working/ss_custom_isic.keras', monitor='val_accuracy', save_best_only=True),
       CSVLogger('/kaggle/working/ss_custom_isic_log.csv')]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST (ISIC-held-out):", model.evaluate(te, verbose=0))

Found 9963 validated image filenames belonging to 6 classes.
Found 2094 validated image filenames belonging to 6 classes.
Found 2091 validated image filenames belonging to 6 classes.
{'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1784716099.181045      59 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784716099.183815      59 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/60


E0000 00:00:1784716102.146875      59 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1784716103.308532     129 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1784716105.472413     129 service.cc:152] XLA service 0x7db161a00050 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784716105.472457     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1784716105.472463     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1784716105.721080     129 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


312/312 [==============================] - 381s 1s/step - loss: 1.9052 - accuracy: 0.2459 - val_loss: 1.9116 - val_accuracy: 0.1394
Epoch 2/60
312/312 [==============================] - 291s 931ms/step - loss: 1.7058 - accuracy: 0.2862 - val_loss: 1.8097 - val_accuracy: 0.2421
Epoch 3/60
312/312 [==============================] - 290s 928ms/step - loss: 1.6082 - accuracy: 0.3275 - val_loss: 3.6692 - val_accuracy: 0.2603
Epoch 4/60
312/312 [==============================] - 289s 925ms/step - loss: 1.5787 - accuracy: 0.3324 - val_loss: 1.5843 - val_accuracy: 0.3734
Epoch 5/60
312/312 [==============================] - 289s 925ms/step - loss: 1.5325 - accuracy: 0.3568 - val_loss: 1.5730 - val_accuracy: 0.3500
Epoch 6/60
312/312 [==============================] - 289s 926ms/step - loss: 1.5397 - accuracy: 0.3680 - val_loss: 1.5272 - val_accuracy: 0.3629
Epoch 7/60
312/312 [==============================] - 290s 930ms/step - loss: 1.4850 - accuracy: 0.3705 - val_loss: 1.4894 - val_accuracy:

In [3]:
# ===== STAGE 11, RUN 2/8: Custom CNN on HAM-only =====
tr, va, te = make_source_gens(h_tr_w, h_va, h_te, None)
print(te.class_indices)

model = build_custom_cnn()
model.compile(Adam(CUSTOM_LR), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint('/kaggle/working/ss_custom_ham.keras', monitor='val_accuracy', save_best_only=True),
       CSVLogger('/kaggle/working/ss_custom_ham_log.csv')]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST (HAM-held-out):", model.evaluate(te, verbose=0))

Found 6789 validated image filenames belonging to 6 classes.
Found 1455 validated image filenames belonging to 6 classes.
Found 1444 validated image filenames belonging to 6 classes.
{'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}
Epoch 1/60


E0000 00:00:1784722957.146406      59 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_1/dropout_4/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


213/213 [==============================] - 221s 1s/step - loss: 1.6717 - accuracy: 0.3722 - val_loss: 2.6097 - val_accuracy: 0.0509
Epoch 2/60
213/213 [==============================] - 168s 790ms/step - loss: 1.4162 - accuracy: 0.4230 - val_loss: 1.8243 - val_accuracy: 0.2055
Epoch 3/60
213/213 [==============================] - 166s 781ms/step - loss: 1.3894 - accuracy: 0.4528 - val_loss: 2.3884 - val_accuracy: 0.2227
Epoch 4/60
213/213 [==============================] - 169s 795ms/step - loss: 1.3487 - accuracy: 0.4609 - val_loss: 1.0844 - val_accuracy: 0.5677
Epoch 5/60
213/213 [==============================] - 167s 782ms/step - loss: 1.2794 - accuracy: 0.4765 - val_loss: 1.0558 - val_accuracy: 0.5512
Epoch 6/60
213/213 [==============================] - 168s 789ms/step - loss: 1.3029 - accuracy: 0.4771 - val_loss: 1.5120 - val_accuracy: 0.2502
Epoch 7/60
213/213 [==============================] - 168s 788ms/step - loss: 1.2444 - accuracy: 0.4830 - val_loss: 1.0707 - val_accuracy:

In [ ]:
# ===== STAGE 11, RUN 3/8: MobileNetV2 on ISIC-only (Phase 1 + Phase 2, continuous log) =====
tr, va, te = make_source_gens(i_tr_w, i_va, i_te, mob_pre)
print(te.class_indices)

model, base = build_pretrained(MobileNetV2)
LOG_FILE = '/kaggle/working/ss_mob_isic_log.csv'

# ---- Phase 1: head only, backbone frozen ----
base.trainable = False
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(LOG_FILE, append=False)], verbose=1)
print("PHASE 1 sanity check (NOT final, backbone still frozen):", model.evaluate(te, verbose=0))

# ---- Phase 2: full fine-tune (RECOMPILE after unfreeze) ----
base.trainable = True
model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint('/kaggle/working/ss_mob_isic.keras', monitor='val_accuracy', save_best_only=True),
       CSVLogger(LOG_FILE, append=True)]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST (ISIC-held-out):", model.evaluate(te, verbose=0))

Found 9963 validated image filenames belonging to 6 classes.
Found 2094 validated image filenames belonging to 6 classes.
Found 2091 validated image filenames belonging to 6 classes.
{'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}
9406464/9406464 [==============================] - 1s 0us/step
Epoch 1/10
312/312 [==============================] - 228s 721ms/step - loss: 1.5911 - accuracy: 0.4382 - val_loss: 1.1733 - val_accuracy: 0.5535
Epoch 2/10
312/312 [==============================] - 228s 732ms/step - loss: 1.2001 - accuracy: 0.5045 - val_loss: 1.3531 - val_accuracy: 0.4938
Epoch 3/10
312/312 [==============================] - 228s 730ms/step - loss: 1.0938 - accuracy: 0.5282 - val_loss: 1.1840 - val_accuracy: 0.5664
Epoch 4/10
312/312 [==============================] - 224s 717ms/step - loss: 1.0217 - accuracy: 0.5464 - val_loss: 1.1559 - val_accuracy: 0.5583
Epoch 5/10
312/312 [==============================] - 222s 713ms/step - loss: 1.0112 - accuracy: 0.551

In [1]:
# ===== FULL REBUILD (new session) + STAGE 11 SETUP =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

# --- config ---
ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES   = 6
IMG_SIZE   = 224
BATCH_SIZE = 32
CUSTOM_LR     = 1e-3
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print("Config loaded")

# --- Stage 1: data build ---
meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']

isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)
print(f"ISIC {len(isic):,} | HAM {len(ham):,} | Pooled {len(isic)+len(ham):,}  (expect 14148 / 9688 / 23836)")

# --- Stage 2: per-source splits ---
def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()
    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))
    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")
    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")
assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")
print(f"i_tr {len(i_tr):,} | i_va {len(i_va):,} | i_te {len(i_te):,}  (expect ~9963 / 2094 / 2091)")
print(f"h_tr {len(h_tr):,} | h_va {len(h_va):,} | h_te {len(h_te):,}  (expect ~6789 / 1455 / 1444)")

# --- model builders ---
def build_custom_cnn(num_classes=6, shape=(224,224,3)):
    m = Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])
    return m

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

# --- Stage 11 setup: per-source class weights, generic generator factory ---
def add_sample_weights(train_src):
    cls = np.array(FINAL_CLASSES)
    cw = compute_class_weight('balanced', classes=cls, y=train_src['label'])
    w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
    train_src = train_src.copy()
    train_src['sample_weight'] = train_src['label'].map(w_map)
    print("  class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})
    return train_src

print("\nISIC-only class weights:")
i_tr_w = add_sample_weights(i_tr)
print("HAM-only class weights:")
h_tr_w = add_sample_weights(h_tr)

def make_source_gens(train_src, val_src, test_src, preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_src, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_src, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_src, shuffle=False, **common)
    return tr, va, te

print("\n===== Rebuild + Stage 11 setup complete. Ready for Run 4/8 =====")

2026-07-23 04:19:46.501780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784780386.749191      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784780386.814023      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784780387.389843      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784780387.389885      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784780387.389887      58 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded
ISIC 14,148 | HAM 9,688 | Pooled 23,836  (expect 14148 / 9688 / 23836)
ISIC lesion-leakage check: PASS
HAM lesion-leakage check: PASS
i_tr 9,963 | i_va 2,094 | i_te 2,091  (expect ~9963 / 2094 / 2091)
h_tr 6,789 | h_va 1,455 | h_te 1,444  (expect ~6789 / 1455 / 1444)

ISIC-only class weights:
  class_weight: {np.str_('bcc'): np.float64(0.852), np.str_('bkl'): np.float64(1.522), np.str_('df'): np.float64(18.049), np.str_('melanoma'): np.float64(0.677), np.str_('nevus'): np.float64(0.386), np.str_('vasc'): np.float64(22.747)}
HAM-only class weights:
  class_weight: {np.str_('bcc'): np.float64(3.017), np.str_('bkl'): np.float64(1.466), np.str_('df'): np.float64(13.969), np.str_('melanoma'): np.float64(1.477), np.str_('nevus'): np.float64(0.241), np.str_('vasc'): np.float64(11.093)}

===== Rebuild + Stage 11 setup complete. Ready for Run 4/8 =====


In [2]:
# ===== STAGE 11, RUN 4/8: MobileNetV2 on HAM-only (Phase 1 + Phase 2, continuous log) =====
tr, va, te = make_source_gens(h_tr_w, h_va, h_te, mob_pre)
print(te.class_indices)

model, base = build_pretrained(MobileNetV2)
LOG_FILE = '/kaggle/working/ss_mob_ham_log.csv'

# ---- Phase 1: head only, backbone frozen ----
base.trainable = False
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(LOG_FILE, append=False)], verbose=1)
print("PHASE 1 sanity check (NOT final, backbone still frozen):", model.evaluate(te, verbose=0))

# ---- Phase 2: full fine-tune (RECOMPILE after unfreeze) ----
base.trainable = True
model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint('/kaggle/working/ss_mob_ham.keras', monitor='val_accuracy', save_best_only=True),
       CSVLogger(LOG_FILE, append=True)]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
print("TEST (HAM-held-out):", model.evaluate(te, verbose=0))

Found 6789 validated image filenames belonging to 6 classes.
Found 1455 validated image filenames belonging to 6 classes.
Found 1444 validated image filenames belonging to 6 classes.
{'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1784780545.083310      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784780545.089247      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 0s 0us/step
Epoch 1/10


I0000 00:00:1784780551.343103     145 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1784780553.715358     145 service.cc:152] XLA service 0x7e2890c47660 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784780553.715394     145 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1784780553.715398     145 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1784780553.997790     145 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


213/213 [==============================] - 188s 855ms/step - loss: 1.4650 - accuracy: 0.4577 - val_loss: 1.2049 - val_accuracy: 0.5615
Epoch 2/10
213/213 [==============================] - 126s 593ms/step - loss: 1.0336 - accuracy: 0.5801 - val_loss: 1.0893 - val_accuracy: 0.5842
Epoch 3/10
213/213 [==============================] - 124s 584ms/step - loss: 0.9878 - accuracy: 0.6067 - val_loss: 1.0002 - val_accuracy: 0.6131
Epoch 4/10
213/213 [==============================] - 127s 596ms/step - loss: 0.8806 - accuracy: 0.6272 - val_loss: 1.0109 - val_accuracy: 0.6117
Epoch 5/10
213/213 [==============================] - 128s 600ms/step - loss: 0.8251 - accuracy: 0.6434 - val_loss: 1.0230 - val_accuracy: 0.6055
Epoch 6/10
213/213 [==============================] - 126s 590ms/step - loss: 0.8026 - accuracy: 0.6572 - val_loss: 1.0870 - val_accuracy: 0.5849
Epoch 7/10
213/213 [==============================] - 126s 593ms/step - loss: 0.7544 - accuracy: 0.6525 - val_loss: 0.7970 - val_accura

In [1]:
import os

files_to_check = [
    'ss_mob_ham.keras', 'ss_mob_ham_log.csv',
]

print("Contents of /kaggle/working/:")
print(os.listdir('/kaggle/working'))
print()

for f in files_to_check:
    p = f'/kaggle/working/{f}'
    exists = os.path.exists(p)
    size = f"{os.path.getsize(p):,} bytes" if exists else "MISSING"
    print(f"{'FOUND ' if exists else 'MISSING'}  {f:30}  {size}")

Contents of /kaggle/working/:
['.virtual_documents']

MISSING  ss_mob_ham.keras                MISSING
MISSING  ss_mob_ham_log.csv              MISSING


In [2]:
# ===== FULL REBUILD (new session) + STAGE 11 SETUP =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
print("Imports ready")

ISIC_IMG_DIR  = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ISIC_CSV      = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_META     = '/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv'
HAM_IMG_DIR_1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMG_DIR_2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_CSV       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES   = 6
IMG_SIZE   = 224
BATCH_SIZE = 32
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print("Config loaded")

meta = pd.read_csv(ISIC_META)
meta['prefix'] = meta['lesion_id'].astype(str).str.extract(r'^([A-Za-z]+)')[0]
isic_keep_ids = set(meta.loc[meta['prefix'] != 'HAM', 'image'])
meta_lesion_map = meta.set_index('image')['lesion_id']

isic = pd.read_csv(ISIC_CSV)
isic = isic[isic['image'].isin(isic_keep_ids)].copy()
dx_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
row_max = isic[dx_cols].max(axis=1)
assert (row_max == 1).all(), f"{(row_max != 1).sum()} ISIC rows have no positive label"
isic['raw_label'] = isic[dx_cols].idxmax(axis=1)
ISIC_MAP = {'MEL':'melanoma','NV':'nevus','BCC':'bcc','BKL':'bkl',
            'DF':'df','VASC':'vasc','AK':None,'SCC':None}
isic['label'] = isic['raw_label'].map(ISIC_MAP)
isic['image_path'] = isic['image'].apply(lambda x: os.path.join(ISIC_IMG_DIR, x + '.jpg'))
isic['source'] = 'isic2019'
isic['lesion_id'] = isic['image'].map(meta_lesion_map)
isic = isic[isic['image_path'].apply(os.path.exists)]
isic = isic[isic['label'].notna()]
isic = isic[['image_path','label','source','lesion_id']].reset_index(drop=True)

ham = pd.read_csv(HAM_CSV)
HAM_MAP = {'mel':'melanoma','nv':'nevus','bcc':'bcc','bkl':'bkl',
           'df':'df','vasc':'vasc','akiec':None}
ham['label'] = ham['dx'].map(HAM_MAP)
def find_ham(i):
    for f in [HAM_IMG_DIR_1, HAM_IMG_DIR_2]:
        p = os.path.join(f, i + '.jpg')
        if os.path.exists(p): return p
    return None
ham['image_path'] = ham['image_id'].apply(find_ham)
ham['source'] = 'ham10000'
ham = ham[ham['image_path'].notna()]
ham = ham[ham['label'].notna()]
ham = ham[['image_path','label','source','lesion_id']].reset_index(drop=True)
print(f"ISIC {len(isic):,} | HAM {len(ham):,} | Pooled {len(isic)+len(ham):,}  (expect 14148 / 9688 / 23836)")

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_lesion_level(df, rs=42, tag="HAM"):
    lg = df.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag=f"{tag} first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag=f"{tag} second split")
    pick = lambda ids: df[df['lesion_id'].isin(ids['lesion_id'])]
    return (pick(l_tr), pick(l_va), pick(l_te),
            set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))

def split_isic_mixed(df, rs=42):
    linked = df[df['lesion_id'].notna()].copy()
    unlinked = df[df['lesion_id'].isna()].copy()
    lg = linked.groupby('lesion_id')['label'].first().reset_index()
    l_tr, l_tmp = safe_split(lg, 'label', 0.30, rs, tag="ISIC linked first split")
    l_va, l_te  = safe_split(l_tmp, 'label', 0.50, rs, tag="ISIC linked second split")
    pick = lambda ids: linked[linked['lesion_id'].isin(ids['lesion_id'])]
    lin_tr, lin_va, lin_te = pick(l_tr), pick(l_va), pick(l_te)
    lesion_sets = (set(l_tr['lesion_id']), set(l_va['lesion_id']), set(l_te['lesion_id']))
    u_tr, u_tmp = safe_split(unlinked, 'label', 0.30, rs, tag="ISIC unlinked first split")
    u_va, u_te  = safe_split(u_tmp, 'label', 0.50, rs, tag="ISIC unlinked second split")
    tr = pd.concat([lin_tr, u_tr], ignore_index=True)
    va = pd.concat([lin_va, u_va], ignore_index=True)
    te = pd.concat([lin_te, u_te], ignore_index=True)
    return tr, va, te, lesion_sets

i_tr, i_va, i_te, isic_lesion_sets = split_isic_mixed(isic)
h_tr, h_va, h_te, s_tr, s_va, s_te = split_lesion_level(ham, tag="HAM")
assert isic_lesion_sets[0].isdisjoint(isic_lesion_sets[2]) and isic_lesion_sets[0].isdisjoint(isic_lesion_sets[1]) and isic_lesion_sets[1].isdisjoint(isic_lesion_sets[2]), "ISIC LEAKAGE"
assert s_tr.isdisjoint(s_te) and s_tr.isdisjoint(s_va) and s_va.isdisjoint(s_te), "HAM LEAKAGE"
print("ISIC lesion-leakage check: PASS")
print("HAM lesion-leakage check: PASS")
print(f"i_tr {len(i_tr):,} | i_va {len(i_va):,} | i_te {len(i_te):,}  (expect ~9963 / 2094 / 2091)")
print(f"h_tr {len(h_tr):,} | h_va {len(h_va):,} | h_te {len(h_te):,}  (expect ~6789 / 1455 / 1444)")

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def add_sample_weights(train_src):
    cls = np.array(FINAL_CLASSES)
    cw = compute_class_weight('balanced', classes=cls, y=train_src['label'])
    w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
    train_src = train_src.copy()
    train_src['sample_weight'] = train_src['label'].map(w_map)
    print("  class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})
    return train_src

print("\nISIC-only class weights:")
i_tr_w = add_sample_weights(i_tr)
print("HAM-only class weights:")
h_tr_w = add_sample_weights(h_tr)

def make_source_gens(train_src, val_src, test_src, preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_src, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_src, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_src, shuffle=False, **common)
    return tr, va, te

print("\n===== Rebuild + Stage 11 setup complete. Ready for Runs 4-8 =====")

2026-07-23 06:52:06.561743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784789526.946790      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784789527.076016      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784789528.029948      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784789528.029993      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784789528.029996      58 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded
ISIC 14,148 | HAM 9,688 | Pooled 23,836  (expect 14148 / 9688 / 23836)
ISIC lesion-leakage check: PASS
HAM lesion-leakage check: PASS
i_tr 9,963 | i_va 2,094 | i_te 2,091  (expect ~9963 / 2094 / 2091)
h_tr 6,789 | h_va 1,455 | h_te 1,444  (expect ~6789 / 1455 / 1444)

ISIC-only class weights:
  class_weight: {np.str_('bcc'): np.float64(0.852), np.str_('bkl'): np.float64(1.522), np.str_('df'): np.float64(18.049), np.str_('melanoma'): np.float64(0.677), np.str_('nevus'): np.float64(0.386), np.str_('vasc'): np.float64(22.747)}
HAM-only class weights:
  class_weight: {np.str_('bcc'): np.float64(3.017), np.str_('bkl'): np.float64(1.466), np.str_('df'): np.float64(13.969), np.str_('melanoma'): np.float64(1.477), np.str_('nevus'): np.float64(0.241), np.str_('vasc'): np.float64(11.093)}

===== Rebuild + Stage 11 setup complete. Ready for Runs 4-8 =====


In [ ]:
# ===== STAGE 11, RUNS 4-8: MobileNetV2/HAM, EfficientNetB0 x2, ResNet50 x2 =====
RUNS = [
    ('mob', 'ham', MobileNetV2, mob_pre, h_tr_w, h_va, h_te),
    ('eff', 'isic', EfficientNetB0, eff_pre, i_tr_w, i_va, i_te),
    ('eff', 'ham', EfficientNetB0, eff_pre, h_tr_w, h_va, h_te),
    ('res', 'isic', ResNet50, res_pre, i_tr_w, i_va, i_te),
    ('res', 'ham', ResNet50, res_pre, h_tr_w, h_va, h_te),
]

run_results = {}
for arch_name, src_name, arch_class, prep_fn, train_src, val_src, test_src in RUNS:
    tag = f"{arch_name}_{src_name}"
    print(f"\n{'='*70}\nSTARTING: {tag}\n{'='*70}")
    try:
        tr, va, te = make_source_gens(train_src, val_src, test_src, prep_fn)
        print(te.class_indices)

        model, base = build_pretrained(arch_class)
        log_file = f'/kaggle/working/ss_{tag}_log.csv'

        base.trainable = False
        model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
        model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(log_file, append=False)], verbose=1)
        phase1_eval = model.evaluate(te, verbose=0)
        print(f"PHASE 1 sanity ({tag}, backbone frozen):", phase1_eval)

        base.trainable = True
        model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
        cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
               ModelCheckpoint(f'/kaggle/working/ss_{tag}.keras', monitor='val_accuracy', save_best_only=True),
               CSVLogger(log_file, append=True)]
        model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)
        test_eval = model.evaluate(te, verbose=0)
        print(f"TEST ({tag}, {src_name}-held-out):", test_eval)

        run_results[tag] = dict(status='SUCCESS', phase1=phase1_eval, test=test_eval)
        del model, base; gc.collect(); tf.keras.backend.clear_session()

    except Exception as e:
        print(f"\n!!! RUN {tag} FAILED: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
        run_results[tag] = dict(status='FAILED', error=str(e))
        gc.collect(); tf.keras.backend.clear_session()
        continue

print(f"\n\n{'='*70}\nRUNS 4-8 SUMMARY\n{'='*70}")
for tag, r in run_results.items():
    if r['status'] == 'SUCCESS':
        print(f"{tag:12} SUCCESS  test_acc={r['test'][1]:.4f}")
    else:
        print(f"{tag:12} FAILED   {r['error']}")


STARTING: mob_ham
Found 6789 validated image filenames belonging to 6 classes.
Found 1455 validated image filenames belonging to 6 classes.
Found 1444 validated image filenames belonging to 6 classes.
{'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1784789700.730866      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784789700.733866      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 0s 0us/step
Epoch 1/10


I0000 00:00:1784789708.234123     142 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1784789711.677956     141 service.cc:152] XLA service 0x7891c87d09d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784789711.678003     141 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1784789711.678007     141 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1784789712.097236     141 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


213/213 [==============================] - 214s 969ms/step - loss: 1.4650 - accuracy: 0.4577 - val_loss: 1.2049 - val_accuracy: 0.5615
Epoch 2/10
213/213 [==============================] - 136s 637ms/step - loss: 1.0336 - accuracy: 0.5801 - val_loss: 1.0893 - val_accuracy: 0.5842
Epoch 3/10
213/213 [==============================] - 133s 623ms/step - loss: 0.9878 - accuracy: 0.6067 - val_loss: 1.0002 - val_accuracy: 0.6131
Epoch 4/10
213/213 [==============================] - 132s 621ms/step - loss: 0.8806 - accuracy: 0.6272 - val_loss: 1.0109 - val_accuracy: 0.6117
Epoch 5/10
213/213 [==============================] - 131s 613ms/step - loss: 0.8251 - accuracy: 0.6434 - val_loss: 1.0230 - val_accuracy: 0.6055
Epoch 6/10
213/213 [==============================] - 131s 615ms/step - loss: 0.8026 - accuracy: 0.6572 - val_loss: 1.0870 - val_accuracy: 0.5849
Epoch 7/10
213/213 [==============================] - 130s 611ms/step - loss: 0.7544 - accuracy: 0.6525 - val_loss: 0.7970 - val_accura

...

312/312 [==============================] - 367s 1s/step - loss: 2.9852 - accuracy: 0.4377 - val_loss: 1.5678 - val_accuracy: 0.4661
Epoch 2/60
312/312 [==============================] - 309s 992ms/step - loss: 1.7827 - accuracy: 0.4579 - val_loss: 1.5852 - val_accuracy: 0.4604
Epoch 3/60
312/312 [==============================] - 306s 982ms/step - loss: 1.4342 - accuracy: 0.4673 - val_loss: 1.5527 - val_accuracy: 0.4656
Epoch 4/60
312/312 [==============================] - 303s 970ms/step - loss: 1.2232 - accuracy: 0.4815 - val_loss: 1.5204 - val_accuracy: 0.4809
Epoch 5/60
312/312 [==============================] - 307s 984ms/step - loss: 1.1291 - accuracy: 0.4951 - val_loss: 1.4666 - val_accuracy: 0.4914
Epoch 6/60
312/312 [==============================] - 305s 979ms/step - loss: 1.0561 - accuracy: 0.5112 - val_loss: 1.4188 - val_accuracy: 0.5043
Epoch 7/60
312/312 [==============================] - 302s 969ms/step - loss: 1.0041 - accuracy: 0.5332 - val_loss: 1.3517 - val_accuracy:

E0000 00:00:1784814700.149095      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


213/213 [==============================] - 143s 645ms/step - loss: 1.3943 - accuracy: 0.4811 - val_loss: 1.5263 - val_accuracy: 0.3601
Epoch 2/10
213/213 [==============================] - 135s 636ms/step - loss: 1.0423 - accuracy: 0.5649 - val_loss: 1.0653 - val_accuracy: 0.5897
Epoch 3/10
213/213 [==============================] - 135s 633ms/step - loss: 0.9454 - accuracy: 0.5840 - val_loss: 0.9572 - val_accuracy: 0.6117
Epoch 4/10
213/213 [==============================] - 135s 634ms/step - loss: 0.9003 - accuracy: 0.5999 - val_loss: 0.7870 - val_accuracy: 0.6770
Epoch 5/10
213/213 [==============================] - 134s 631ms/step - loss: 0.8393 - accuracy: 0.6428 - val_loss: 0.8328 - val_accuracy: 0.6577
Epoch 6/10
213/213 [==============================] - 134s 630ms/step - loss: 0.7765 - accuracy: 0.6335 - val_loss: 1.1113 - val_accuracy: 0.5512
Epoch 7/10
213/213 [==============================] - 134s 631ms/step - loss: 0.7640 - accuracy: 0.6590 - val_loss: 1.0170 - val_accura

E0000 00:00:1784816079.283107      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


213/213 [==============================] - 220s 830ms/step - loss: 2.5997 - accuracy: 0.5046 - val_loss: 1.4912 - val_accuracy: 0.4564
Epoch 2/60
213/213 [==============================] - 173s 813ms/step - loss: 1.5845 - accuracy: 0.4949 - val_loss: 1.2562 - val_accuracy: 0.5278
Epoch 3/60
213/213 [==============================] - 171s 803ms/step - loss: 1.2413 - accuracy: 0.5067 - val_loss: 1.1961 - val_accuracy: 0.5375
Epoch 4/60
213/213 [==============================] - 172s 806ms/step - loss: 1.0943 - accuracy: 0.5259 - val_loss: 1.1400 - val_accuracy: 0.5574
Epoch 5/60
213/213 [==============================] - 170s 800ms/step - loss: 0.9966 - accuracy: 0.5504 - val_loss: 1.1216 - val_accuracy: 0.5718
Epoch 6/60
213/213 [==============================] - 172s 806ms/step - loss: 0.9148 - accuracy: 0.5815 - val_loss: 1.0689 - val_accuracy: 0.5897
Epoch 7/60
213/213 [==============================] - 172s 807ms/step - loss: 0.8553 - accuracy: 0.5970 - val_loss: 1.0473 - val_accura